# EEG-RL pipeline

Runs end to end from the stored files in the `allpickle` Kaggle dataset (pkl, csv, checkpoints). Anything already stored is loaded directly, anything missing is built and cached in `/kaggle/working/cache`. The last cells list the newly created files and zip them, so they can be added to the dataset for the next run.

## Setup

In [2]:
import os, re, math, time, pickle, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.optimize import minimize
from scipy.stats import pearsonr, ttest_1samp, ttest_rel
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/kaggle/input")
WORK       = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
CACHE_DIR  = WORK / "cache"
MODEL_DIR  = WORK / "models"
CACHE_DIR.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)

ALLOW_TRAIN   = True     # False = never train the CNN, raise an error if a checkpoint is missing
FORCE_REBUILD = set()    # e.g. {"m2_fits.csv"} rebuilds just that file
REGIONS = ["frontal", "central", "parieto_occipital"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# duplicate downloads like "file (1).csv" are treated as "file.csv"
_DUP = re.compile(r"\s*\(\d+\)(?=\.[^.]+$)")
def _norm(fname): return _DUP.sub("", fname)
def _dup_rank(p):
    m = re.search(r"\((\d+)\)\.[^.]+$", p.name)
    return int(m.group(1)) if m else 0

def _where(p): return "working" if str(p).startswith(str(WORK)) else "INPUT"

NEW_FILES = []          # files created in this session (to upload later)
def _track(p, upload=True):
    p = Path(p)
    if p not in INDEX.setdefault(_norm(p.name), []): INDEX[_norm(p.name)].append(p)
    if upload and p not in NEW_FILES: NEW_FILES.append(p)

def _read(p):
    if p.suffix == ".csv": return pd.read_csv(p)
    with open(p, "rb") as f: return pickle.load(f)

def _write(obj, p):
    if p.suffix == ".csv": obj.to_csv(p, index=False)
    else:
        with open(p, "wb") as f: pickle.dump(obj, f)

def cached(name, builder, legacy=()):
    # load the stored file if it exists, otherwise build it and save to the cache dir
    # legacy = older alternative names
    if name not in FORCE_REBUILD:
        for n in (name, *legacy):
            p = find(n)
            if p is None: continue
            try:
                obj = _read(p); print(f"loaded {n:<44} <- {_where(p)}"); return obj
            except Exception as e:
                print(f"could not read {n} ({e}), rebuilding")
    print(f"building {name} (not stored)")
    obj = builder()
    out = CACHE_DIR / name
    _write(obj, out); _track(out)
    print(f"saved {name}")
    return obj

def export_new_files():
    # list the files created in this run and zip them for upload to the allpickle dataset
    files = [p for p in NEW_FILES if p.exists()]
    print("=" * 70)
    if not files:
        print("No new files were created, everything was loaded from input.")
        return
    zp = WORK / "new_files_to_upload.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files: z.write(p, arcname=p.name)
    print("New files from this run (add them to the allpickle dataset):")
    for p in files: print(f"   {p.name:<48} {p.stat().st_size/1e6:8.2f} MB")
    print(f"\nZip: {zp}  ({zp.stat().st_size/1e6:.1f} MB), download, unzip and upload to the dataset.")

device: cuda
GPU: Tesla T4


## Load stored data

In [3]:
import tarfile, hashlib, shutil

# look for stored files everywhere, archives (zip/tar) are extracted and scanned as well
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "/kaggle", "/content", "/mnt",
                "/data", "/workspace", os.path.expanduser("~"), os.getcwd()]
SKIP_DIRS   = {"proc", "sys", "dev", "__pycache__", ".git", ".ipynb_checkpoints", "node_modules"}
EXTRACT_DIR = WORK / "_extracted"
ARCHIVE_EXT = (".zip", ".tar", ".tar.gz", ".tgz", ".tar.bz2", ".tar.xz")
MAX_ARCHIVE_GB = 2      # larger archives are not extracted

def _walk_all(roots):
    seen = set()
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, dirs, files in os.walk(root, followlinks=True):
            real = os.path.realpath(dp)
            if real in seen:
                dirs[:] = []; continue
            seen.add(real)
            dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
            for f in files: yield Path(dp) / f

def _extract(p):
    out = EXTRACT_DIR / hashlib.md5(str(p).encode()).hexdigest()[:8]
    if out.exists(): return out
    try:
        if p.stat().st_size > MAX_ARCHIVE_GB * 1e9:
            print(f"  skip (large archive): {p}"); return None
        out.mkdir(parents=True, exist_ok=True)
        if p.name.lower().endswith(".zip"):
            with zipfile.ZipFile(p) as z: z.extractall(out)
        else:
            with tarfile.open(p) as t: t.extractall(out)
        print(f"  extracted: {p}")
        return out
    except Exception as e:
        print(f"  extract failed {p}: {e}")
        shutil.rmtree(out, ignore_errors=True)
        return None

def build_index():
    idx, done, roots = {}, set(), list(SEARCH_ROOTS)
    while True:
        new_arch = []
        for p in _walk_all(roots):
            idx.setdefault(_norm(p.name), []).append(p)
            in_work_output = str(p).startswith(str(WORK)) and not str(p).startswith(str(EXTRACT_DIR))
            if p.name.lower().endswith(ARCHIVE_EXT) and p not in done and not in_work_output:
                new_arch.append(p)
        roots = []
        for a in new_arch:                 # archives can contain archives, so loop again
            done.add(a)
            out = _extract(a)
            if out: roots.append(out)
        if not roots: break
    return idx

INDEX = build_index()
_LOWER = {n.lower(): n for n in INDEX}
print(f"indexed {sum(len(v) for v in INDEX.values())} files ({len(INDEX)} unique names)")

def find(name):
    # returns a Path or None, working dir first then input, plain name first then (1), (2)
    key = _norm(name)
    hits = INDEX.get(key) or INDEX.get(_LOWER.get(key.lower(), ""), [])
    if not hits: return None
    return sorted(hits, key=lambda p: (0 if str(p).startswith(str(WORK)) else 1, _dup_rank(p), len(str(p))))[0]

def _need(name):
    # builder for cached() when the file has to exist already
    def _b(): raise FileNotFoundError(f"'{name}' not found in input, check the allpickle dataset")
    return _b

  extracted: /root/.julia/registries/General.tar.gz
  extracted: /root/.julia/packages/JSON/bn3ui/test/jsonchecker.tar
  extracted: /root/.julia/packages/JSON/bn3ui/test/JSONTestSuite.tar
indexed 92349 files (23656 unique names)


In [4]:
# region tensors
def load_subjects():
    files = {}
    for name in INDEX:
        m = re.fullmatch(r"sub-s(\d+)\.pkl", name, re.I)
        if m: files[int(m.group(1))] = find(name)
    data = {}
    for n in sorted(files):
        with open(files[n], "rb") as f: data[f"sub-s{n}"] = pickle.load(f)
    return data

all_subjects_data = load_subjects()
assert len(all_subjects_data) == 23, f"Expected 23 subject pkl files, found {len(all_subjects_data)}"
_s0 = next(iter(all_subjects_data.values()))
print(f"region tensors: {len(all_subjects_data)} subjects | keys={list(_s0.keys())}")
print("  shapes:", {k: v.shape for k, v in _s0["region_tensors"].items()})

# behavior trials
final_pipeline_data = cached("final_validated_trials.pkl", _need("final_validated_trials.pkl"))

region tensors: 23 subjects | keys=['region_tensors', 'rpe', 'reward', 'times']
  shapes: {'frontal': (560, 8, 1025), 'central': (560, 8, 1025), 'parieto_occipital': (560, 9, 1025)}
loaded final_validated_trials.pkl                   <- INPUT


In [5]:
# RL models (M1/M2/M3): fit + RPE
PARAMS  = {"m1": ["alpha", "beta"],
           "m2": ["alpha_gain", "alpha_loss", "beta"],
           "m3": ["alpha0", "kappa", "eta", "beta"]}
BOUNDS  = {"m1": [(0.001, 0.999), (0.01, 20)],
           "m2": [(0.001, 0.999), (0.001, 0.999), (0.01, 20)],
           "m3": [(0.0, 1.0), (0.0, 1.0), (0.001, 0.999), (0.01, 20)]}
RPE_COL = {"m1": "rpe", "m2": "rpe_m2", "m3": "rpe_m3"}

def _sig(x): return 1.0 / (1.0 + math.exp(-max(min(x, 500.0), -500.0)))
def _arr(df): return (np.asarray(df["choice"]) == "B").astype(int), np.asarray(df["reward"], dtype=float)

def run_model(model, p, c, r):
    # returns (nll, rpe array), Q[0] = A, Q[1] = B
    Q = [0.5, 0.5]; A = [0.5, 0.5]; nll = 0.0; rpes = np.empty(len(c))
    for t in range(len(c)):
        ch = c[t]
        nll -= math.log(max(_sig(p[-1] * (Q[ch] - Q[1 - ch])), 1e-6))     # beta is always the last param
        rpe = r[t] - Q[ch]; rpes[t] = rpe
        if model == "m1":
            Q[ch] += p[0] * rpe
        elif model == "m2":
            Q[ch] += (p[0] if r[t] == 1 else p[1]) * rpe
        else:
            a_t = min(max(p[0] + p[1] * A[ch], 0.001), 0.999)
            Q[ch] += a_t * rpe
            A[ch] = p[2] * abs(rpe) + (1 - p[2]) * A[ch]
    return nll, rpes

def _in_domain(model, p):
    if model == "m1": return 0 < p[0] < 1 and p[1] > 0
    if model == "m2": return 0 < p[0] < 1 and 0 < p[1] < 1 and p[2] > 0
    return 0 <= p[0] <= 1 and 0 <= p[1] <= 1 and 0 < p[2] < 1 and p[3] > 0

def _x0(model, rng):
    if model == "m1": return [rng.uniform(0.05, 0.95), rng.uniform(0.1, 10.0)]
    if model == "m2": return [rng.uniform(0.05, 0.95), rng.uniform(0.05, 0.95), rng.uniform(0.1, 10.0)]
    return [rng.uniform(0.05, 0.5), rng.uniform(0.05, 0.95), rng.uniform(0.05, 0.95), rng.uniform(0.1, 10.0)]

def robust_fit(model, trials_df, n_starts=10, seed=42):
    c, r = _arr(trials_df); rng = np.random.default_rng(seed)
    best_fun, best_x = np.inf, None
    for _ in range(n_starts):
        res = minimize(lambda p: run_model(model, p, c, r)[0] if _in_domain(model, p) else 1e10,
                       x0=_x0(model, rng), method="Nelder-Mead", bounds=BOUNDS[model])
        if res.success and res.fun < best_fun: best_fun, best_x = res.fun, res.x
    return best_x, best_fun

def build_fits(model):
    rows, t0, n = [], time.time(), len(final_pipeline_data)
    for i, (key, trials) in enumerate(final_pipeline_data.items(), 1):
        subj, block = key.rsplit("_", 1)
        x, nll = robust_fit(model, trials)
        chance = len(trials) * np.log(2)
        rows.append({"subject": subj, "block": block, "n_trials": len(trials), **dict(zip(PARAMS[model], x)),
                     "nll": nll, "chance_nll": chance, "above_chance": nll < chance - 5})
        print(f"  [{model}] {i}/{n} {key} nll={nll:.2f} ({time.time()-t0:.0f}s)")
    return pd.DataFrame(rows)

def build_rpe_pipeline(model, fits_df):
    out = {}
    for key, trials in final_pipeline_data.items():
        subj, block = key.rsplit("_", 1)
        row = fits_df[(fits_df["subject"] == subj) & (fits_df["block"] == block)]
        if row.empty: continue
        c, r = _arr(trials)
        _, rpes = run_model(model, [row[k].values[0] for k in PARAMS[model]], c, r)
        t = trials.copy(); t[RPE_COL[model]] = rpes; out[key] = t
    return out

final_rw_df             = cached("rw_fits.csv",                  lambda: build_fits("m1"))
final_pipeline_with_rpe = cached("final_trials_with_rpe.pkl",    lambda: build_rpe_pipeline("m1", final_rw_df))
m2_fit_df               = cached("m2_fits.csv",                  lambda: build_fits("m2"))
final_pipeline_m2       = cached("final_trials_with_rpe_m2.pkl", lambda: build_rpe_pipeline("m2", m2_fit_df))
m3_fit_df               = cached("m3_fits.csv",                  lambda: build_fits("m3"))
final_pipeline_m3       = cached("final_trials_with_rpe_m3.pkl", lambda: build_rpe_pipeline("m3", m3_fit_df))
print(f"\nblocks above chance → M1 {final_rw_df['above_chance'].sum()}/{len(final_rw_df)} | "
      f"M2 {m2_fit_df['above_chance'].sum()}/{len(m2_fit_df)} | M3 {m3_fit_df['above_chance'].sum()}/{len(m3_fit_df)}")

loaded rw_fits.csv                                  <- INPUT
loaded final_trials_with_rpe.pkl                    <- INPUT
loaded m2_fits.csv                                  <- INPUT
loaded final_trials_with_rpe_m2.pkl                 <- INPUT
loaded m3_fits.csv                                  <- INPUT
loaded final_trials_with_rpe_m3.pkl                 <- INPUT

blocks above chance → M1 35/46 | M2 39/46 | M3 35/46


In [6]:
# trial order alignment, RPE lookups (M1/M2/M3) and region features
def combined_trials(subj, pipeline=None):
    # same order as when the region tensors were built: REW + PUN sorted by feedback_sample
    pipeline = final_pipeline_with_rpe if pipeline is None else pipeline
    parts = [pipeline[k] for k in (f"{subj}_REW", f"{subj}_PUN") if k in pipeline]
    return pd.concat(parts, ignore_index=True).sort_values("feedback_sample").reset_index(drop=True)

SUBJECTS = list(all_subjects_data.keys())
RPE = {"m1": {s: np.asarray(all_subjects_data[s]["rpe"]) for s in SUBJECTS},
       "m2": {s: combined_trials(s, final_pipeline_m2)["rpe_m2"].values for s in SUBJECTS},
       "m3": {s: combined_trials(s, final_pipeline_m3)["rpe_m3"].values for s in SUBJECTS}}

bad = [(s, len(all_subjects_data[s]["rpe"]), len(RPE["m2"][s]), len(RPE["m3"][s])) for s in SUBJECTS
       if not (len(all_subjects_data[s]["rpe"]) == len(RPE["m2"][s]) == len(RPE["m3"][s]))]
assert not bad, f"trial count mismatch (subject, n_m1, n_m2, n_m3): {bad}"
diff = max(np.abs(RPE["m1"][s] - combined_trials(s)["rpe"].values).max() for s in SUBJECTS)
print(f"23 subjects: M1/M2/M3 trial counts match | stored-M1-RPE vs pipeline-M1-RPE max|diff| = {diff:.2e}")
if diff > 1e-6:
    print("warning: rpe in the region tensors does not match final_trials_with_rpe.pkl, they may come from different runs")

def build_features():
    dfs = []
    for s in SUBJECTS:
        d = all_subjects_data[s]; t = np.asarray(d["times"]); sf = 1.0 / (t[1] - t[0])
        i0, i1 = int(round((0.25 - t[0]) * sf)), int(round((0.50 - t[0]) * sf))      # FRN/theta window 0.25–0.5s
        f = pd.DataFrame({reg: d["region_tensors"][reg][:, :, i0:i1].mean(axis=(1, 2)) for reg in REGIONS})
        f["subject"] = s
        f["trial_num"] = combined_trials(s)["trial_num"].values[:len(f)]
        f["rpe"] = d["rpe"]; f["reward"] = d["reward"]
        dfs.append(f)
    return pd.concat(dfs, ignore_index=True)

feature_df = cached("region_features_rpe.csv", build_features)
print(f"feature_df: {len(feature_df)} rows, {feature_df['subject'].nunique()} subjects")

23 subjects: M1/M2/M3 trial counts match | stored-M1-RPE vs pipeline-M1-RPE max|diff| = 0.00e+00
loaded region_features_rpe.csv                      <- INPUT
feature_df: 12880 rows, 23 subjects


## CNN + leave-one-subject-out decoding

In [7]:
class EEGRegionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class SimpleEEGCNN(nn.Module):
    def __init__(self, n_channels, n_times):
        super().__init__()
        self.conv1 = nn.Conv1d(n_channels, 16, kernel_size=15, padding=7)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=9, padding=4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x = self.act(self.conv1(x)); x = self.act(self.conv2(x))
        return self.fc(self.pool(x).squeeze(-1)).squeeze(-1)

def fit_predict_fold(train_X, train_y, test_X, ckpt_name, n_epochs=30, lr=1e-3):
    # load the checkpoint if it exists, otherwise train and save it. returns (test_pred, status)
    model = SimpleEEGCNN(train_X.shape[1], train_X.shape[2]).to(device)
    ck = find(ckpt_name)
    if ck is not None:
        model.load_state_dict(torch.load(ck, map_location=device)); status = "ckpt"
    else:
        if not ALLOW_TRAIN: raise RuntimeError(f"{ckpt_name} is not stored and ALLOW_TRAIN=False")
        loader = DataLoader(EEGRegionDataset(train_X, train_y), batch_size=32, shuffle=True)
        opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.MSELoss(); model.train()
        for _ in range(n_epochs):
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
        out = MODEL_DIR / ckpt_name
        torch.save(model.state_dict(), out); _track(out, upload=False); status = "trained"
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(test_X, dtype=torch.float32).to(device)).cpu().numpy()
    return pred, status

def ckpt_name(region, tag, subj, seed):     # matches the old checkpoint names
    if tag == "m1": return f"loso_{region}_{subj}.pt" if seed == 0 else f"loso_{region}_{subj}_seed{seed}.pt"
    return f"loso_{region}_{tag}_{subj}_seed{seed}.pt"

def loso_predict(region, tag="m1", seed=0):
    torch.manual_seed(seed)
    y = RPE[tag]; rows, status_count, t0 = [], {}, time.time()
    for i, te in enumerate(SUBJECTS, 1):
        tr = [s for s in SUBJECTS if s != te]
        Xtr = np.concatenate([all_subjects_data[s]["region_tensors"][region] for s in tr])
        ytr = np.concatenate([y[s] for s in tr])
        Xte = all_subjects_data[te]["region_tensors"][region]
        mu, sd = Xtr.mean(), Xtr.std() + 1e-8
        pred, status = fit_predict_fold((Xtr - mu) / sd, ytr, (Xte - mu) / sd, ckpt_name(region, tag, te, seed))
        status_count[status] = status_count.get(status, 0) + 1
        if status == "trained": print(f"    fold {i}/{len(SUBJECTS)} {te} trained ({time.time()-t0:.0f}s)")
        rt = combined_trials(te)["rt"].values
        n = min(len(pred), len(rt))
        rows.append(pd.DataFrame({"subject": te, "trial": np.arange(n), "actual_rpe": y[te][:n], "predicted_rpe": pred[:n],
                                  "reward": all_subjects_data[te]["reward"][:n], "rt": rt[:n]}))
    print(f"    [{region}/{tag}/seed{seed}] folds: {status_count}")
    return pd.concat(rows, ignore_index=True)

def get_predictions(region, tag="m1", seed=0):
    legacy = []
    if tag == "m1" and seed == 0:
        legacy = ["central_predictions_with_rt.csv"] if region == "central" else [f"{region}_predictions_with_reward.csv"]
    return cached(f"pred_{region}_{tag}_seed{seed}.csv", lambda: loso_predict(region, tag, seed), legacy=legacy)

# stats helpers
def partial_corr(x, y, z):
    r_xy, r_xz, r_yz = pearsonr(x, y)[0], pearsonr(x, z)[0], pearsonr(y, z)[0]
    den = np.sqrt(max(0.0, 1 - r_xz**2) * max(0.0, 1 - r_yz**2))
    return np.nan if den == 0 else (r_xy - r_xz * r_yz) / den

def interaction_table(pred_df):
    rows = []
    for subj, sdf in pred_df.groupby("subject"):
        if len(sdf) < 20 or sdf["reward"].nunique() < 2: continue
        sdf = sdf.copy(); sdf["reward"] = sdf["reward"].astype(float)
        m = smf.ols("actual_rpe ~ predicted_rpe * reward", data=sdf).fit()
        if "predicted_rpe:reward" in m.params:
            rows.append({"subject": subj, "beta": m.params["predicted_rpe:reward"], "p": m.pvalues["predicted_rpe:reward"]})
    return pd.DataFrame(rows)

def interaction_test(pred_df):
    tb = interaction_table(pred_df); b = tb["beta"].dropna().values
    t, p = ttest_1samp(b, 0)
    return dict(mean_beta=b.mean(), t=t, p=p, n=len(b), n_sig=int((tb["p"] < 0.05).sum()))

def reward_confound(pred_df, label, show_rows=True):
    rows = []
    for subj, sdf in pred_df.groupby("subject"):
        if len(sdf) < 20 or sdf["reward"].std() == 0 or sdf["predicted_rpe"].std() == 0: continue
        pr, ac, rw = sdf["predicted_rpe"].values, sdf["actual_rpe"].values, sdf["reward"].values
        def within(mask):
            s = sdf[mask]
            ok = len(s) > 5 and s["predicted_rpe"].std() > 0 and s["actual_rpe"].std() > 0
            return pearsonr(s["predicted_rpe"], s["actual_rpe"])[0] if ok else np.nan
        rows.append({"subject": subj, "raw_corr": pearsonr(pr, ac)[0], "corr_with_reward": pearsonr(pr, rw)[0],
                     "partial_corr": partial_corr(pr, ac, rw),
                     "within_win_corr": within(sdf["reward"] == 1), "within_loss_corr": within(sdf["reward"] == 0)})
    df = pd.DataFrame(rows)
    print(f"\n{'=' * 25} {label.upper()} {'=' * 25}")
    if show_rows: print(df.round(3).to_string(index=False))
    for col in ["raw_corr", "corr_with_reward", "partial_corr", "within_win_corr", "within_loss_corr"]:
        v = df[col].dropna().values
        if len(v) > 1:
            t, p = ttest_1samp(v, 0); print(f"Mean {col:18s}: {v.mean(): .4f}  (t={t:.3f}, p={p:.4e}, n={len(v)})")
    v = df["partial_corr"].dropna().clip(-0.999, 0.999).values
    if len(v) > 1:
        t, p = ttest_1samp(np.arctanh(v), 0.0); print(f"Fisher z-test (partial > 0): t={t:.3f}, p={p:.4e}")
    return df
print("CNN + LOSO + stats helpers ready")

CNN + LOSO + stats helpers ready


## RPE vs region amplitude (group level, FDR)

In [8]:
rows = []
for subj in feature_df["subject"].unique():
    sdf = feature_df[feature_df["subject"] == subj]
    for reg in REGIONS:
        m0 = smf.ols(f"{reg} ~ rpe", data=sdf).fit()
        m1 = smf.ols(f"{reg} ~ rpe + reward", data=sdf).fit()
        rows.append({"subject": subj, "region": reg, "beta": m0.params["rpe"], "beta_ctrl": m1.params["rpe"]})
per_subj = pd.DataFrame(rows)

def group_ttest(col):
    out = []
    for reg in REGIONS:
        b = per_subj.loc[per_subj["region"] == reg, col].values; t, p = ttest_1samp(b, 0)
        out.append({"region": reg, "mean_beta": b.mean(), "n": len(b), "t": t, "p": p})
    d = pd.DataFrame(out); d["p_fdr"] = multipletests(d["p"], method="fdr_bh")[1]; d["sig_fdr"] = d["p_fdr"] < 0.05
    return d

print("=== amplitude ~ rpe (FDR) ===");            print(group_ttest("beta").to_string(index=False))
print("\n=== amplitude ~ rpe + reward (FDR) ===");  print(group_ttest("beta_ctrl").to_string(index=False))
coll = [abs(pearsonr(g["rpe"], g["reward"])[0]) for _, g in feature_df.groupby("subject")]
print(f"\nMean |RPE-reward correlation| across subjects: {np.mean(coll):.3f}")

=== amplitude ~ rpe (FDR) ===
           region    mean_beta  n        t        p    p_fdr  sig_fdr
          frontal 1.365171e-06 23 3.135941 0.004803 0.014410     True
          central 5.091077e-07 23 2.603358 0.016224 0.024337     True
parieto_occipital 1.902211e-07 23 1.831842 0.080550 0.080550    False

=== amplitude ~ rpe + reward (FDR) ===
           region     mean_beta  n         t        p    p_fdr  sig_fdr
          frontal  2.168276e-06 23  1.998967 0.058115 0.058115    False
          central  1.020661e-06 23  2.088638 0.048526 0.058115    False
parieto_occipital -2.753038e-07 23 -2.080160 0.049367 0.058115    False

Mean |RPE-reward correlation| across subjects: 0.860


## CNN decoding (M1 RPE, seed 0) and confound checks

In [9]:
region_pred_dfs = {reg: get_predictions(reg, "m1", 0) for reg in ["central", "frontal", "parieto_occipital"]}

# --- motor confound (central): RT control ---
cdf = region_pred_dfs["central"]
if "rt" in cdf.columns:
    rows = []
    for subj, sdf in cdf.dropna(subset=["rt"]).groupby("subject"):
        if len(sdf) < 20 or sdf["rt"].std() == 0 or sdf["predicted_rpe"].std() == 0: continue
        pr, ac, rt = sdf["predicted_rpe"].values, sdf["actual_rpe"].values, sdf["rt"].values
        rows.append({"subject": subj, "raw_corr": pearsonr(pr, ac)[0], "corr_with_RT": pearsonr(pr, rt)[0],
                     "partial_corr_RT_controlled": partial_corr(pr, ac, rt)})
    motor_df = pd.DataFrame(rows)
    print("=== Motor-confound (central): mean over subjects ===")
    print(motor_df.drop(columns="subject").mean().round(4).to_string())
else:
    print("no rt column in central predictions, skipping motor check")

# --- reward-valence confound (3 regions) ---
conf_tables = {reg: reward_confound(df, reg, show_rows=False) for reg, df in region_pred_dfs.items()}

# --- FDR on partial_corr + interaction (predicted_rpe × reward) ---
regs = list(region_pred_dfs)
p_part = [ttest_1samp(conf_tables[r]["partial_corr"].dropna(), 0)[1] for r in regs]
fdr_part = pd.DataFrame({"region": regs, "p_raw": p_part, "p_fdr": multipletests(p_part, method="fdr_bh")[1]})
fdr_part["significant"] = fdr_part["p_fdr"] < 0.05
print("\n=== FDR: graded RPE decodable after removing reward-valence? (partial_corr) ===")
print(fdr_part.to_string(index=False))

inter = {r: interaction_test(region_pred_dfs[r]) for r in regs}
int_df = pd.DataFrame([{"region": r, **v} for r, v in inter.items()])
int_df["p_fdr"] = multipletests(int_df["p"], method="fdr_bh")[1]; int_df["significant"] = int_df["p_fdr"] < 0.05
print("\n=== Interaction: predicted_rpe × reward → actual_rpe (per-subject β, group t-test, FDR) ===")
print(int_df.round(4).to_string(index=False))

loaded central_predictions_with_rt.csv              <- INPUT
loaded frontal_predictions_with_reward.csv          <- INPUT
loaded pred_parieto_occipital_m1_seed0.csv          <- INPUT
=== Motor-confound (central): mean over subjects ===
raw_corr                      0.0662
corr_with_RT                 -0.0290
partial_corr_RT_controlled    0.0676

========================= CENTRAL =========================
Mean raw_corr          :  0.0662  (t=5.154, p=3.6307e-05, n=23)
Mean corr_with_reward  :  0.0663  (t=4.485, p=1.8435e-04, n=23)
Mean partial_corr      :  0.0185  (t=1.584, p=1.2751e-01, n=23)
Mean within_win_corr   :  0.0013  (t=0.090, p=9.2877e-01, n=23)
Mean within_loss_corr  :  0.0389  (t=2.100, p=4.7461e-02, n=23)
Fisher z-test (partial > 0): t=1.583, p=1.2759e-01

========================= FRONTAL =========================
Mean raw_corr          :  0.0797  (t=5.700, p=9.8581e-06, n=23)
Mean corr_with_reward  :  0.1006  (t=6.492, p=1.5700e-06, n=23)
Mean partial_corr      : -0.0098

## Multi-seed robustness (PO interaction)

In [10]:
seed_rows = []
for seed in [0, 1, 2, 3]:
    r = interaction_test(get_predictions("parieto_occipital", "m1", seed))
    seed_rows.append({"seed": seed, **r})
seed_df = pd.DataFrame(seed_rows)
print("=== PO interaction β across CNN training seeds ===")
print(seed_df.round(4).to_string(index=False))
b = seed_df["mean_beta"].values
print(f"\nMean β={b.mean():.4f} | SD={b.std():.4f} | CV={b.std()/abs(b.mean()):.3f}")
print(f"seeds with β<0 and p<0.05: {int(((seed_df['mean_beta'] < 0) & (seed_df['p'] < 0.05)).sum())}/{len(seed_df)}")

loaded pred_parieto_occipital_m1_seed0.csv          <- INPUT
loaded pred_parieto_occipital_m1_seed1.csv          <- INPUT
loaded pred_parieto_occipital_m1_seed2.csv          <- INPUT
loaded pred_parieto_occipital_m1_seed3.csv          <- INPUT
=== PO interaction β across CNN training seeds ===
 seed  mean_beta       t      p  n  n_sig
    0    -0.4287 -3.4184 0.0025 23      4
    1    -0.3387 -2.6342 0.0151 23      3
    2    -0.4135 -3.0827 0.0054 23      3
    3    -0.5690 -2.4386 0.0233 23      2

Mean β=-0.4375 | SD=0.0832 | CV=0.190
seeds with β<0 and p<0.05: 4/4


## Behavior model comparison (M1 vs M2 vs M3, AIC)

In [11]:
def _m(df): return {(r.subject, r.block): r for r in df.itertuples()}
d1, d2, d3 = _m(final_rw_df), _m(m2_fit_df), _m(m3_fit_df)
rows = []
for key in final_pipeline_data:
    subj, block = key.rsplit("_", 1); k = (subj, block)
    if k not in d1 or k not in d2 or k not in d3: continue
    aic = {"M1": 4 + 2 * d1[k].nll, "M2": 6 + 2 * d2[k].nll, "M3": 8 + 2 * d3[k].nll}
    rows.append({"subject": subj, "block": block, **{f"aic_{m.lower()}": v for m, v in aic.items()}, "winner": min(aic, key=aic.get)})
three_way_df = pd.DataFrame(rows)
print(f"subject-blocks compared: {len(three_way_df)}")
print("winner counts:\n" + three_way_df["winner"].value_counts().to_string())

dlt = three_way_df["aic_m1"] - three_way_df["aic_m2"]
t, p = ttest_1samp(dlt, 0)
print(f"\nH1 (M1 vs M2): mean ΔAIC (M1−M2, +ve = M2 better) = {dlt.mean():.3f}, t={t:.3f}, p={p:.4f}, mean|Δ|={dlt.abs().mean():.3f}")
print("=> " + ("H1 SUPPORTED (model mimicry present)" if dlt.abs().mean() < 4 else "H1 NOT clearly supported"))
for a, b_ in [("aic_m1", "aic_m2"), ("aic_m1", "aic_m3"), ("aic_m2", "aic_m3")]:
    m_ = (three_way_df[a] - three_way_df[b_]).abs().mean()
    print(f"{a} vs {b_}: mean|ΔAIC|={m_:.3f} => {'mimicry present' if m_ < 4 else 'mimicry weak/absent'}")

subject-blocks compared: 46
winner counts:
winner
M2    27
M1    15
M3     4

H1 (M1 vs M2): mean ΔAIC (M1−M2, +ve = M2 better) = 8.684, t=3.604, p=0.0008, mean|Δ|=9.859
=> H1 NOT clearly supported
aic_m1 vs aic_m2: mean|ΔAIC|=9.859 => mimicry weak/absent
aic_m1 vs aic_m3: mean|ΔAIC|=4.396 => mimicry weak/absent
aic_m2 vs aic_m3: mean|ΔAIC|=12.085 => mimicry weak/absent


## Parameter recovery (synthetic data)

In [12]:
RANGES = {"m1": [(.05, .95), (.5, 15)], "m2": [(.05, .95), (.05, .95), (.5, 15)],
          "m3": [(.05, .5), (.05, .95), (.05, .95), (.5, 15)]}

def simulate(model, p, n_trials, seed):
    rng = np.random.default_rng(seed); Q = [0.5, 0.5]; A = [0.5, 0.5]; ch_l, rw_l = [], []
    for _ in range(n_trials):
        ch = 0 if rng.random() < _sig(p[-1] * (Q[0] - Q[1])) else 1
        rw = 1 if rng.random() < (0.7 if ch == 0 else 0.3) else 0
        rpe = rw - Q[ch]
        if model == "m1": Q[ch] += p[0] * rpe
        elif model == "m2": Q[ch] += (p[0] if rw == 1 else p[1]) * rpe
        else:
            Q[ch] += min(max(p[0] + p[1] * A[ch], 0.001), 0.999) * rpe
            A[ch] = p[2] * abs(rpe) + (1 - p[2]) * A[ch]
        ch_l.append("A" if ch == 0 else "B"); rw_l.append(rw)
    return pd.DataFrame({"choice": ch_l, "reward": rw_l})

def build_recovery(n_subj=30, n_trials=280):
    rng = np.random.default_rng(123); rows = []
    for mi, model in enumerate(["m1", "m2", "m3"], 1):
        for i in range(n_subj):
            true = [rng.uniform(*b) for b in RANGES[model]]
            synth = simulate(model, true, n_trials, seed=mi * 1000 + i)
            rec, _ = robust_fit(model, synth, n_starts=10, seed=mi * 1000 + i)
            row = {"model": model.upper(), "synth_id": i}
            for k, tv, rv in zip(PARAMS[model], true, rec): row[f"true_{k}"] = tv; row[f"rec_{k}"] = rv
            rows.append(row)
        print(f"  recovery {model.upper()} done")
    return pd.DataFrame(rows)

recovery_df = cached("param_recovery.csv", build_recovery)
print("\n=== RECOVERY: true vs recovered correlation (r>0.7 good, <0.3 poor) ===")
for model in ["m1", "m2", "m3"]:
    sub = recovery_df[recovery_df["model"] == model.upper()]
    print(f"--- {model.upper()} ---")
    for k in PARAMS[model]:
        v = sub[[f"true_{k}", f"rec_{k}"]].dropna()
        if len(v) > 2:
            r_ = pearsonr(v.iloc[:, 0], v.iloc[:, 1])[0]
            print(f"  {k:12s} r={r_:.3f}  {'good' if r_ > 0.7 else ('weak' if r_ > 0.3 else 'poor')}")

loaded param_recovery.csv                           <- INPUT

=== RECOVERY: true vs recovered correlation (r>0.7 good, <0.3 poor) ===
--- M1 ---
  alpha        r=0.960  good
  beta         r=0.833  good
--- M2 ---
  alpha_gain   r=0.719  good
  alpha_loss   r=0.882  good
  beta         r=0.694  weak
--- M3 ---
  alpha0       r=0.557  weak
  kappa        r=0.520  weak
  eta          r=0.428  weak
  beta         r=0.913  good


## M1 vs official M5 RPE (optional)

In [13]:
pp = find("participants.tsv")
if pp is None:
    print("participants.tsv not found in input, skipping M5 comparison")
else:
    participants = pd.read_csv(pp, sep="\t")
    def rpe_m5(trials, a, d):
        c, r = _arr(trials); Q = [0.5, 0.5]; out = np.empty(len(c))
        for t in range(len(c)):
            ch = c[t]; rpe = r[t] - Q[ch]; out[t] = rpe
            Q[ch] += a * rpe; Q[1 - ch] += d * (0.5 - Q[1 - ch])
        return out
    pipe_m5 = {}
    for key, trials in final_pipeline_data.items():
        subj, block = key.rsplit("_", 1)
        row = participants[participants["participant_id"] == subj]
        if row.empty: continue
        t = trials.copy()
        t["rpe_m5"] = rpe_m5(trials, row[f"M5_{block}_alpha"].values[0], row[f"M5_{block}_decay"].values[0])
        pipe_m5[key] = t
    rows = []
    for subj in feature_df["subject"].unique():
        sdf = feature_df[feature_df["subject"] == subj].copy()
        sdf["rpe_m5"] = combined_trials(subj, pipe_m5)["rpe_m5"].values[:len(sdf)]
        rows.append({"subject": subj, "corr_m1_m5": pearsonr(sdf["rpe"], sdf["rpe_m5"])[0],
                     "m1_r2": smf.ols("frontal ~ rpe", data=sdf).fit().rsquared,
                     "m5_r2": smf.ols("frontal ~ rpe_m5", data=sdf).fit().rsquared})
    cmp_df = pd.DataFrame(rows)
    t, p = ttest_rel(cmp_df["m1_r2"], cmp_df["m5_r2"])
    print(f"M5 better in {(cmp_df['m5_r2'] > cmp_df['m1_r2']).sum()}/{len(cmp_df)} subjects | mean R²: M1={cmp_df['m1_r2'].mean():.4f}, M5={cmp_df['m5_r2'].mean():.4f}")
    print(f"paired t-test (frontal R²): t={t:.3f}, p={p:.4f}")

participants.tsv not found in input, skipping M5 comparison


## RPE correlation across models

In [14]:
cm = pd.DataFrame([{"subject": s,
                    "r_m1_m2": pearsonr(RPE["m1"][s], RPE["m2"][s])[0],
                    "r_m1_m3": pearsonr(RPE["m1"][s], RPE["m3"][s])[0],
                    "r_m2_m3": pearsonr(RPE["m2"][s], RPE["m3"][s])[0]} for s in SUBJECTS])
print(cm.drop(columns="subject").describe().loc[["mean", "std", "min", "max"]].round(3).to_string())

      r_m1_m2  r_m1_m3  r_m2_m3
mean    0.938    0.970    0.906
std     0.094    0.047    0.106
min     0.642    0.852    0.642
max     1.000    1.000    1.000


## PO region: M1 vs M2 vs M3 (interaction, FDR)

In [15]:
try:
    po = {tag: get_predictions("parieto_occipital", tag, 0) for tag in ("m1", "m2", "m3")}
    res = {tag.upper(): interaction_test(df) for tag, df in po.items()}

    fdr = pd.DataFrame({"model": list(res), "mean_beta": [r["mean_beta"] for r in res.values()],
                        "t": [r["t"] for r in res.values()], "p_raw": [r["p"] for r in res.values()],
                        "subjects_sig": [f'{r["n_sig"]}/{r["n"]}' for r in res.values()]})
    fdr["p_fdr"] = multipletests(fdr["p_raw"], method="fdr_bh")[1]
    fdr["significant_fdr"] = fdr["p_fdr"] < 0.05
    print("=== FDR-corrected: PO interaction across M1, M2, M3 ===")
    print(fdr.round(4).to_string(index=False))

    summ = []
    for tag, df in po.items():
        tb = reward_confound(df, f"PO / {tag.upper()}", show_rows=False)
        summ.append({"model": tag.upper(), "raw_corr_mean": tb["raw_corr"].mean(),
                     "partial_corr_mean": tb["partial_corr"].mean(),
                     "interaction_beta": res[tag.upper()]["mean_beta"],
                     "interaction_p_fdr": fdr.loc[fdr["model"] == tag.upper(), "p_fdr"].values[0]})
    print("\n" + "=" * 70 + "\n=== SUMMARY: PO region, M1 vs M2 vs M3 ===\n" + "=" * 70)
    print(pd.DataFrame(summ).round(4).to_string(index=False))

    sig = fdr.loc[fdr["significant_fdr"], "model"].tolist()
    print(f"\nSignificant after FDR: {sig if sig else 'none'}")
finally:
    export_new_files()

loaded pred_parieto_occipital_m1_seed0.csv          <- INPUT
loaded pred_parieto_occipital_m2_seed0.csv          <- INPUT
loaded pred_parieto_occipital_m3_seed0.csv          <- INPUT
=== FDR-corrected: PO interaction across M1, M2, M3 ===
model  mean_beta       t  p_raw subjects_sig  p_fdr  significant_fdr
   M1    -0.4287 -3.4184 0.0025         4/23 0.0041             True
   M2    -0.0251 -0.5108 0.6146         1/23 0.6146            False
   M3    -0.4881 -3.3774 0.0027         2/23 0.0041             True

========================= PO / M1 =========================
Mean raw_corr          :  0.0467  (t=3.401, p=2.5670e-03, n=23)
Mean corr_with_reward  :  0.0584  (t=4.485, p=1.8457e-04, n=23)
Mean partial_corr      : -0.0083  (t=-0.705, p=4.8792e-01, n=23)
Mean within_win_corr   : -0.0321  (t=-1.966, p=6.2036e-02, n=23)
Mean within_loss_corr  :  0.0265  (t=1.778, p=8.9261e-02, n=23)
Fisher z-test (partial > 0): t=-0.705, p=4.8828e-01

========================= PO / M2 ===============

In [16]:
# PO interaction across 4 seeds (0-3) for M1/M2/M3
try:
    all_seed = []
    for tag in ["m1", "m2", "m3"]:
        for seed in [0, 1, 2, 3]:
            r = interaction_test(get_predictions("parieto_occipital", tag, seed))
            all_seed.append({"model": tag.upper(), "seed": seed, **r})
    seed_all = pd.DataFrame(all_seed)
    print("=== Results per seed ===")
    print(seed_all.round(4).to_string(index=False))

    rows = []
    for m, g in seed_all.groupby("model"):
        rows.append({"model": m,
                     "mean_beta": g["mean_beta"].mean(), "sd_beta": g["mean_beta"].std(),
                     "min_beta": g["mean_beta"].min(), "max_beta": g["mean_beta"].max(),
                     "seeds (β<0 & p<0.05)": f"{int(((g['mean_beta'] < 0) & (g['p'] < 0.05)).sum())}/{len(g)}",
                     "median_p": g["p"].median()})
    print("\n=== Model-wise robustness summary ===")
    print(pd.DataFrame(rows).round(4).to_string(index=False))
finally:
    export_new_files()

loaded pred_parieto_occipital_m1_seed0.csv          <- INPUT
loaded pred_parieto_occipital_m1_seed1.csv          <- INPUT
loaded pred_parieto_occipital_m1_seed2.csv          <- INPUT
loaded pred_parieto_occipital_m1_seed3.csv          <- INPUT
loaded pred_parieto_occipital_m2_seed0.csv          <- INPUT
loaded pred_parieto_occipital_m2_seed1.csv          <- INPUT
loaded pred_parieto_occipital_m2_seed2.csv          <- INPUT
loaded pred_parieto_occipital_m2_seed3.csv          <- INPUT
loaded pred_parieto_occipital_m3_seed0.csv          <- INPUT
loaded pred_parieto_occipital_m3_seed1.csv          <- INPUT
loaded pred_parieto_occipital_m3_seed2.csv          <- INPUT
loaded pred_parieto_occipital_m3_seed3.csv          <- INPUT
=== Results per seed ===
model  seed  mean_beta       t      p  n  n_sig
   M1     0    -0.4287 -3.4184 0.0025 23      4
   M1     1    -0.3387 -2.6342 0.0151 23      3
   M1     2    -0.4135 -3.0827 0.0054 23      3
   M1     3    -0.5690 -2.4386 0.0233 23      2
   

## Cross-valence decoding (PO)

In [17]:
def combined_trials_blocked(subj):
    parts = []
    for k, blk in [(f"{subj}_REW", "REW"), (f"{subj}_PUN", "PUN")]:
        if k in final_pipeline_with_rpe:
            p = final_pipeline_with_rpe[k].copy(); p["__block__"] = blk; parts.append(p)
    return pd.concat(parts, ignore_index=True).sort_values("feedback_sample").reset_index(drop=True)

BLOCK = {s: combined_trials_blocked(s)["__block__"].values for s in SUBJECTS}
assert all(len(BLOCK[s]) == len(all_subjects_data[s]["rpe"]) for s in SUBJECTS)
print("BLOCK labels ready")

BLOCK labels ready


In [18]:
def crossval_preds(region, tag, direction, seed=0):
    tr_blk, te_blk = direction.split("->")
    tagd = direction.replace(">", "")
    def build():
        y = RPE[tag]; torch.manual_seed(seed); rows = []
        for te in SUBJECTS:
            Xtr, ytr = [], []
            for s in SUBJECTS:
                if s == te: continue
                m = BLOCK[s] == tr_blk
                if m.sum() < 10: continue
                Xtr.append(all_subjects_data[s]["region_tensors"][region][m]); ytr.append(y[s][m])
            Xtr = np.concatenate(Xtr); ytr = np.concatenate(ytr)
            tm = BLOCK[te] == te_blk
            Xte = all_subjects_data[te]["region_tensors"][region][tm]
            mu, sd = Xtr.mean(), Xtr.std() + 1e-8
            pred, st = fit_predict_fold((Xtr - mu) / sd, ytr, (Xte - mu) / sd,
                                        f"crossval_{region}_{tag}_{tagd}_{te}_seed{seed}.pt")
            rows.append(pd.DataFrame({"subject": te, "direction": direction, "trial": np.arange(int(tm.sum())),
                                      "actual_rpe": y[te][tm], "predicted_rpe": pred,
                                      "reward": np.asarray(all_subjects_data[te]["reward"])[tm]}))
        return pd.concat(rows, ignore_index=True)
    return cached(f"crossval_pred_{region}_{tag}_{tagd}_seed{seed}.csv", build)

try:
    out = []
    for tag in ["m1", "m2", "m3"]:
        for d in ["REW->PUN", "PUN->REW"]:
            df = crossval_preds("parieto_occipital", tag, d)
            raw, par = [], []
            for s, g in df.groupby("subject"):
                p_, a_, r_ = g["predicted_rpe"].values, g["actual_rpe"].values, g["reward"].values.astype(float)
                if p_.std() == 0 or a_.std() == 0: continue
                raw.append(pearsonr(p_, a_)[0])
                if r_.std() > 0: par.append(partial_corr(p_, a_, r_))
            par = np.array(par); par = par[~np.isnan(par)]
            t1, p1 = ttest_1samp(raw, 0); t2, p2 = ttest_1samp(par, 0)
            out.append({"model": tag.upper(), "direction": d,
                        "raw_corr": np.mean(raw), "p_raw": p1,
                        "partial_corr(reward ctrl)": par.mean(), "p_partial": p2, "n": len(par)})
    cv = pd.DataFrame(out)
    cv["p_fdr_raw"] = multipletests(cv["p_raw"], method="fdr_bh")[1]
    cv["p_fdr_partial"] = multipletests(cv["p_partial"], method="fdr_bh")[1]
    print("=== Cross-valence, PO: raw vs reward-controlled ===")
    print(cv.round(4).to_string(index=False))
finally:
    export_new_files()

loaded crossval_pred_parieto_occipital_m1_REW-PUN_seed0.csv <- INPUT
loaded crossval_pred_parieto_occipital_m1_PUN-REW_seed0.csv <- INPUT
loaded crossval_pred_parieto_occipital_m2_REW-PUN_seed0.csv <- INPUT
loaded crossval_pred_parieto_occipital_m2_PUN-REW_seed0.csv <- INPUT
loaded crossval_pred_parieto_occipital_m3_REW-PUN_seed0.csv <- INPUT
loaded crossval_pred_parieto_occipital_m3_PUN-REW_seed0.csv <- INPUT
=== Cross-valence, PO: raw vs reward-controlled ===
model direction  raw_corr  p_raw  partial_corr(reward ctrl)  p_partial  n  p_fdr_raw  p_fdr_partial
   M1  REW->PUN    0.0474 0.0226                     0.0110     0.4759 23     0.0678         0.9528
   M1  PUN->REW    0.0428 0.0041                    -0.0209     0.2353 23     0.0246         0.9528
   M2  REW->PUN    0.0283 0.0830                    -0.0046     0.8203 23     0.0996         0.9528
   M2  PUN->REW    0.0339 0.0478                    -0.0009     0.9528 23     0.0927         0.9528
   M3  REW->PUN    0.0322 0.1329  

In [20]:
# H4: cross-region combination (M1, seed 0), averages the stored predictions, no training
import itertools
AB = {"frontal": "F", "central": "C", "parieto_occipital": "PO"}
P = {}
for r in REGIONS:
    d = get_predictions(r, "m1", 0).copy()
    d["trial"] = d.groupby("subject").cumcount()            # join on (subject, trial), not on row order
    print(f"{r:18s} rows={len(d)} subjects={d['subject'].nunique()} first subjects={list(d['subject'].unique()[:3])}")
    P[r] = d.set_index(["subject", "trial"]).sort_index()
idx = P[REGIONS[0]].index
for r in REGIONS[1:]: idx = idx.intersection(P[r].index)
print(f"common (subject, trial) rows: {len(idx)}")
base = P["frontal"].loc[idx].reset_index()
for r in REGIONS:
    dmax = np.abs(P[r].loc[idx, "actual_rpe"].values - base["actual_rpe"].values).max()
    print(f"  {r:18s} max|Δ actual_rpe| = {dmax:.2e}")
    assert dmax < 1e-6, f"{r}: actual_rpe mismatch, this CSV was built from a different RPE / trial order"
Z = {}
for r in REGIONS:
    v = P[r].loc[idx, "predicted_rpe"].values
    Z[r] = (v - v.mean()) / v.std()

def subj_metrics(df):
    out = {}
    for s, g in df.groupby("subject"):
        p_, a_, r_ = g["predicted_rpe"].values, g["actual_rpe"].values, g["reward"].values.astype(float)
        if p_.std() == 0 or a_.std() == 0 or r_.std() == 0: continue
        out[s] = (pearsonr(p_, a_)[0], partial_corr(p_, a_, r_))
    return out

combos = [(r,) for r in REGIONS] + [c for k in (2, 3) for c in itertools.combinations(REGIONS, k)]
rows, M = [], {}
for c in combos:
    df = base[["subject", "actual_rpe", "reward"]].copy()
    df["predicted_rpe"] = np.mean([Z[r] for r in c], axis=0)
    M[c] = subj_metrics(df)
    raw = np.array([v[0] for v in M[c].values()]); par = np.array([v[1] for v in M[c].values()]); par = par[~np.isnan(par)]
    it = interaction_test(df)
    rows.append({"regions": "+".join(AB[x] for x in c), "raw_corr": raw.mean(), "p_raw": ttest_1samp(raw, 0)[1],
                 "partial_corr": par.mean(), "p_partial": ttest_1samp(par, 0)[1],
                 "interaction_beta": it["mean_beta"], "p_int": it["p"]})
res_h4 = pd.DataFrame(rows)
for col in ["p_raw", "p_partial", "p_int"]:
    res_h4[col + "_fdr"] = multipletests(res_h4[col], method="fdr_bh")[1]
print("\n=== H4: single regions vs combinations (M1, seed 0; FDR across 7 combos) ===")
print(res_h4[["regions", "raw_corr", "p_raw_fdr", "partial_corr", "p_partial_fdr", "interaction_beta", "p_int_fdr"]].round(4).to_string(index=False))

best = max([(r,) for r in REGIONS], key=lambda c: np.mean([v[0] for v in M[c].values()]))
allc = tuple(REGIONS)
common = sorted(set(M[best]) & set(M[allc]))
a = [M[allc][s][0] for s in common]; b = [M[best][s][0] for s in common]
t, p = ttest_rel(a, b)
print(f"\nBest single region (raw): {AB[best[0]]}  ->  raw_corr={np.mean(b):.4f}")
print(f"All three (F+C+PO): raw_corr={np.mean(a):.4f} | paired t-test vs best single: t={t:.3f}, p={p:.4f}, n={len(common)}")
print("H4:", "combining regions improves decoding" if (p < 0.05 and np.mean(a) > np.mean(b)) else "combining regions does not improve decoding")

loaded frontal_predictions_with_reward.csv          <- INPUT
frontal            rows=12880 subjects=23 first subjects=['sub-s25', 'sub-s4', 'sub-s18']
loaded central_predictions_with_rt.csv              <- INPUT
central            rows=12880 subjects=23 first subjects=['sub-s25', 'sub-s4', 'sub-s18']
loaded pred_parieto_occipital_m1_seed0.csv          <- INPUT
parieto_occipital  rows=12880 subjects=23 first subjects=['sub-s1', 'sub-s2', 'sub-s3']
common (subject, trial) rows: 12880
  frontal            max|Δ actual_rpe| = 0.00e+00
  central            max|Δ actual_rpe| = 0.00e+00
  parieto_occipital  max|Δ actual_rpe| = 0.00e+00

=== H4: single regions vs combinations (M1, seed 0; FDR across 7 combos) ===
regions  raw_corr  p_raw_fdr  partial_corr  p_partial_fdr  interaction_beta  p_int_fdr
      F    0.0797     0.0000       -0.0098         0.5183           -0.0091     0.3406
      C    0.0662     0.0000        0.0185         0.5183           -0.0275     0.1377
     PO    0.0467     0.

In [21]:
# subject-level decodability vs behavior (M1, seed 0), needs M from the H4 cell above
from scipy.stats import spearmanr

beh = {}
for s in SUBJECTS:
    ts = [final_pipeline_data[k].sort_values("feedback_sample") for k in (f"{s}_REW", f"{s}_PUN") if k in final_pipeline_data]
    win = np.concatenate([t["reward"].values for t in ts]).mean()
    stay = np.concatenate([t["choice"].values[1:] == t["choice"].values[:-1] for t in ts]).mean()
    f = final_rw_df[final_rw_df["subject"] == s]
    beh[s] = {"win_rate": win, "stay_rate": stay, "alpha_M1": f["alpha"].mean(), "beta_M1": f["beta"].mean()}
beh_df = pd.DataFrame(beh).T
print("=== behavior summary (across subjects) ===")
print(beh_df.describe().loc[["mean", "std", "min", "max"]].round(3).to_string())

rows = []
for r in REGIONS:
    dec = pd.DataFrame(M[(r,)], index=["raw_corr", "partial_corr"]).T
    for dcol in ["raw_corr", "partial_corr"]:
        for bcol in beh_df.columns:
            x = dec[dcol]; y = beh_df.loc[x.index, bcol]
            ok = x.notna() & y.notna()
            rho, p = spearmanr(x[ok], y[ok])
            rows.append({"region": AB[r], "decodability": dcol, "behavior": bcol, "rho": rho, "p": p, "n": int(ok.sum())})
sq5 = pd.DataFrame(rows)
sq5["p_fdr"] = multipletests(sq5["p"], method="fdr_bh")[1]
sq5["sig_fdr"] = sq5["p_fdr"] < 0.05
print(f"\n=== Spearman: decodability vs behavior ({len(sq5)} tests, FDR) ===")
print(sq5.sort_values("p").round(4).to_string(index=False))
print("\nAfter FDR:", f"{int(sq5['sig_fdr'].sum())} significant" if sq5["sig_fdr"].any()
      else f"no relationship is significant (smallest raw p={sq5['p'].min():.3f}, n=23, low power)")

=== behavior summary (across subjects) ===
      win_rate  stay_rate  alpha_M1  beta_M1
mean     0.604      0.800     0.405    3.635
std      0.028      0.076     0.294    3.628
min      0.546      0.629     0.021    0.010
max      0.659      0.910     0.999   12.845

=== Spearman: decodability vs behavior (24 tests, FDR) ===
region decodability  behavior     rho      p  n  p_fdr  sig_fdr
     F     raw_corr   beta_M1 -0.2669 0.2183 23 0.9714    False
     F     raw_corr stay_rate  0.2566 0.2372 23 0.9714    False
     C partial_corr   beta_M1  0.2357 0.2789 23 0.9714    False
    PO     raw_corr   beta_M1 -0.2288 0.2936 23 0.9714    False
     F     raw_corr  win_rate  0.2144 0.3260 23 0.9714    False
    PO     raw_corr  alpha_M1 -0.2065 0.3444 23 0.9714    False
    PO     raw_corr  win_rate  0.1921 0.3799 23 0.9714    False
     C partial_corr stay_rate  0.1498 0.4951 23 0.9714    False
     F partial_corr  win_rate -0.1356 0.5372 23 0.9714    False
     F partial_corr   beta_M1  0

In [22]:
# gradient boosting baseline (EEG window features -> M1 RPE, leave-one-subject-out)
from sklearn.ensemble import HistGradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

AB = {"frontal": "F", "central": "C", "parieto_occipital": "PO"}
WINS = [(0.0, 0.15), (0.15, 0.25), (0.25, 0.35), (0.35, 0.5), (0.5, 0.8)]      # seconds, feedback-locked

def build_feat_matrix():
    F = {}
    for s in SUBJECTS:
        d = all_subjects_data[s]; t = np.asarray(d["times"]); cols = {}
        for reg in REGIONS:
            X = d["region_tensors"][reg]                                          # trials × channels × time
            for a, b in WINS:
                m = (t >= a) & (t < b)
                if m.sum() == 0: continue
                cols[f"{AB[reg]}_{a}-{b}"] = X[:, :, m].mean(axis=(1, 2))
        F[s] = pd.DataFrame(cols)
    return F
FEATS = build_feat_matrix()
print(f"features per trial: {FEATS[SUBJECTS[0]].shape[1]}  (3 regions x {len(WINS)} time windows, no reward/behavior features so no leakage)")

def gb_loso(kind):
    rows = []
    for i, te in enumerate(SUBJECTS, 1):
        tr = [s for s in SUBJECTS if s != te]
        Xtr = pd.concat([FEATS[s] for s in tr]).values; ytr = np.concatenate([RPE["m1"][s] for s in tr])
        if kind == "hgb":
            mdl = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=3, random_state=0)
        else:
            mdl = AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=3), n_estimators=100, learning_rate=0.05, random_state=0)
        mdl.fit(Xtr, ytr); pred = mdl.predict(FEATS[te].values)
        rows.append(pd.DataFrame({"subject": te, "trial": np.arange(len(pred)), "actual_rpe": RPE["m1"][te],
                                  "predicted_rpe": pred, "reward": np.asarray(all_subjects_data[te]["reward"])}))
        if i % 6 == 0: print(f"    [{kind}] fold {i}/{len(SUBJECTS)}")
    return pd.concat(rows, ignore_index=True)

try:
    gb = {k: cached(f"gb_pred_{k}_m1.csv", lambda k=k: gb_loso(k)) for k in ("hgb", "ada")}
    out = []
    for k, df in gb.items():
        tb = reward_confound(df, f"GB / {k}", show_rows=False)
        it = interaction_test(df)
        out.append({"model": k, "raw_corr": tb["raw_corr"].mean(), "p_raw": ttest_1samp(tb["raw_corr"].dropna(), 0)[1],
                    "partial_corr": tb["partial_corr"].mean(), "p_partial": ttest_1samp(tb["partial_corr"].dropna(), 0)[1],
                    "interaction_beta": it["mean_beta"], "p_int": it["p"]})
    gbs = pd.DataFrame(out)
    for c in ["p_raw", "p_partial", "p_int"]: gbs[c + "_fdr"] = multipletests(gbs[c], method="fdr_bh")[1]
    print("\n=== Gradient boosting (EEG features), reward-controlled ===")
    print(gbs[["model", "raw_corr", "p_raw_fdr", "partial_corr", "p_partial_fdr", "interaction_beta", "p_int_fdr"]].round(4).to_string(index=False))
    print("\nCNN (M1, seed 0):  raw_corr  F=%.4f  C=%.4f  PO=%.4f | partial_corr  F=%.4f  C=%.4f  PO=%.4f" % (
        *[conf_tables[r]["raw_corr"].mean() for r in ["frontal", "central", "parieto_occipital"]],
        *[conf_tables[r]["partial_corr"].mean() for r in ["frontal", "central", "parieto_occipital"]]))
finally:
    export_new_files()

features per trial: 15  (3 regions x 5 time windows, no reward/behavior features so no leakage)
loaded gb_pred_hgb_m1.csv                           <- INPUT
loaded gb_pred_ada_m1.csv                           <- INPUT

========================= GB / HGB =========================
Mean raw_corr          :  0.1362  (t=7.201, p=3.2331e-07, n=23)
Mean corr_with_reward  :  0.1498  (t=7.450, p=1.8841e-07, n=23)
Mean partial_corr      :  0.0234  (t=1.664, p=1.1029e-01, n=23)
Mean within_win_corr   :  0.0026  (t=0.163, p=8.7196e-01, n=23)
Mean within_loss_corr  :  0.0469  (t=2.568, p=1.7531e-02, n=23)
Fisher z-test (partial > 0): t=1.664, p=1.1024e-01

========================= GB / ADA =========================
Mean raw_corr          :  0.1059  (t=4.633, p=1.2856e-04, n=23)
Mean corr_with_reward  :  0.1164  (t=4.964, p=5.7502e-05, n=23)
Mean partial_corr      :  0.0219  (t=1.605, p=1.2279e-01, n=23)
Mean within_win_corr   :  0.0190  (t=1.432, p=1.6608e-01, n=23)
Mean within_loss_corr  :  0.023

In [23]:
# GMM clustering of subject-level RL parameters (exploratory, M1 fits)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score
from scipy.stats import mannwhitneyu

piv = final_rw_df.pivot_table(index="subject", columns="block", values="alpha")
par_df = pd.DataFrame({"alpha": final_rw_df.groupby("subject")["alpha"].mean(),
                       "log10_beta": np.log10(final_rw_df.groupby("subject")["beta"].mean()),
                       "alpha_REW_minus_PUN": piv["REW"] - piv["PUN"]}).dropna()
X = StandardScaler().fit_transform(par_df.values)
print(f"subjects: {len(par_df)} | features: {list(par_df.columns)}  (small n, exploratory only)")

bics = {}
for K in range(1, 5):
    gm = GaussianMixture(K, covariance_type="diag", n_init=10, random_state=0).fit(X)
    bics[K] = gm.bic(X)
print("\n=== BIC (lower is better) ===")
print(pd.Series(bics, name="BIC").round(2).to_string())
bestK = min(bics, key=bics.get)
print(f"\nBest K by BIC = {bestK}")

if bestK == 1:
    print("=> no evidence for subgroups (a single cluster fits best), report as a null result.")
else:
    gm = GaussianMixture(bestK, covariance_type="diag", n_init=10, random_state=0).fit(X)
    lab = gm.predict(X); par_df["cluster"] = lab
    print(f"silhouette = {silhouette_score(X, lab):.3f}")
    rng = np.random.default_rng(0); aris = []
    for _ in range(100):                                    # bootstrap stability
        ix = rng.choice(len(X), len(X), replace=True)
        g2 = GaussianMixture(bestK, covariance_type="diag", n_init=5, random_state=0).fit(X[ix])
        aris.append(adjusted_rand_score(lab, g2.predict(X)))
    print(f"bootstrap stability (ARI vs original): mean={np.mean(aris):.3f}  (>0.7 stable, <0.4 unstable)")
    print("\n=== cluster profile (mean parameters) ===")
    print(par_df.groupby("cluster").agg(["mean"]).round(3).assign(n=par_df.groupby("cluster").size()).to_string())
    if bestK == 2 and "M" in globals():
        rows = []
        for r in REGIONS:
            dec = pd.DataFrame(M[(r,)], index=["raw_corr", "partial_corr"]).T
            for col in ["raw_corr", "partial_corr"]:
                a = dec.loc[dec.index.intersection(par_df.index[par_df.cluster == 0]), col].dropna()
                b = dec.loc[dec.index.intersection(par_df.index[par_df.cluster == 1]), col].dropna()
                rows.append({"region": AB[r], "metric": col, "cl0_mean": a.mean(), "cl1_mean": b.mean(),
                             "p": mannwhitneyu(a, b).pvalue, "n0": len(a), "n1": len(b)})
        cl = pd.DataFrame(rows); cl["p_fdr"] = multipletests(cl["p"], method="fdr_bh")[1]
        print("\n=== cluster vs decodability (Mann-Whitney, FDR; exploratory) ===")
        print(cl.round(4).to_string(index=False))

subjects: 23 | features: ['alpha', 'log10_beta', 'alpha_REW_minus_PUN']  (small n, exploratory only)

=== BIC (lower is better) ===
1    214.63
2    169.86
3    147.97
4    155.21

Best K by BIC = 3
silhouette = 0.388
bootstrap stability (ARI vs original): mean=0.271  (>0.7 stable, <0.4 unstable)

=== cluster profile (mean parameters) ===
         alpha log10_beta alpha_REW_minus_PUN   n
          mean       mean                mean    
cluster                                          
0        0.505      1.013              -0.987   1
1        0.044     -2.000              -0.017   2
2        0.436      0.412               0.063  20


In [24]:
# H3: full region x model grid (3 regions x M1/M2/M3, seed 0)
# if the frontal/central M2/M3 predictions are not stored this trains (~45-50 min, once), CSVs are saved afterwards
try:
    rows = []
    for reg in REGIONS:
        for tag in ("m1", "m2", "m3"):
            df = get_predictions(reg, tag, 0)
            m = subj_metrics(df[["subject", "actual_rpe", "predicted_rpe", "reward"]])
            raw = np.array([v[0] for v in m.values()]); par = np.array([v[1] for v in m.values()]); par = par[~np.isnan(par)]
            it = interaction_test(df)
            rows.append({"region": AB[reg], "model": tag.upper(), "raw_corr": raw.mean(), "p_raw": ttest_1samp(raw, 0)[1],
                         "partial_corr": par.mean(), "p_partial": ttest_1samp(par, 0)[1],
                         "interaction_beta": it["mean_beta"], "p_int": it["p"]})
    grid = pd.DataFrame(rows)
    for c in ["p_raw", "p_partial", "p_int"]: grid[c + "_fdr"] = multipletests(grid[c], method="fdr_bh")[1]
    print("=== H3 grid: 3 regions × 3 models (FDR across 9 tests) ===")
    print(grid[["region", "model", "raw_corr", "p_raw_fdr", "partial_corr", "p_partial_fdr", "interaction_beta", "p_int_fdr"]].round(4).to_string(index=False))

    print("\n=== Best model per region (raw_corr / partial_corr) ===")
    for reg in grid["region"].unique():
        g = grid[grid["region"] == reg]
        print(f"  {reg:3s}: raw → {g.loc[g['raw_corr'].idxmax(), 'model']}   partial → {g.loc[g['partial_corr'].idxmax(), 'model']}")
    print("\nH3:", "no partial_corr is significant after FDR, so no model separates from reward in any region"
          if not (grid["p_partial_fdr"] < 0.05).any() else f"partial_corr significant: {grid.loc[grid['p_partial_fdr'] < 0.05, ['region', 'model']].values.tolist()}")
finally:
    export_new_files()

loaded frontal_predictions_with_reward.csv          <- INPUT
loaded pred_frontal_m2_seed0.csv                    <- INPUT
loaded pred_frontal_m3_seed0.csv                    <- INPUT
loaded central_predictions_with_rt.csv              <- INPUT
loaded pred_central_m2_seed0.csv                    <- INPUT
loaded pred_central_m3_seed0.csv                    <- INPUT
loaded pred_parieto_occipital_m1_seed0.csv          <- INPUT
loaded pred_parieto_occipital_m2_seed0.csv          <- INPUT
loaded pred_parieto_occipital_m3_seed0.csv          <- INPUT
=== H3 grid: 3 regions × 3 models (FDR across 9 tests) ===
region model  raw_corr  p_raw_fdr  partial_corr  p_partial_fdr  interaction_beta  p_int_fdr
     F    M1    0.0797     0.0001       -0.0098         0.5796           -0.0883     0.5109
     F    M2    0.0602     0.0002        0.0097         0.5796            0.0122     0.8160
     F    M3    0.0612     0.0025       -0.0096         0.5796           -0.0592     0.6914
     C    M1    0.0662  

In [25]:
# one global BH-FDR pass over the main hypothesis tests from the earlier cells
# (multi-seed robustness and GMM clustering are left out, they are exploratory)

need = {"grid": "H3 grid", "fdr": "PO across M1/M2/M3", "cv": "cross-valence",
        "sq5": "decodability vs behavior", "gbs": "gradient boosting", "res_h4": "H4 region combination"}
missing = [f"{k} ({v})" for k, v in need.items() if k not in globals()]
assert not missing, "Run these cells first:\n" + "\n".join(missing)

rows = []

# H3 grid: 9 region/model combos x 3 metrics = 27 tests
for _, r in grid.iterrows():
    for metric, col in [("raw_corr", "p_raw"), ("partial_corr", "p_partial"), ("interaction_beta", "p_int")]:
        rows.append({"family": "H3_grid", "test": f"{r['region']}/{r['model']}/{metric}", "p": r[col]})

# the PO interaction tests across M1/M2/M3 are already in the H3 grid, skipped to avoid double counting

# cross-valence: raw + partial per model x direction = 12 tests
for _, r in cv.iterrows():
    rows.append({"family": "cross_valence", "test": f"{r['model']}/{r['direction']}/raw", "p": r["p_raw"]})
    rows.append({"family": "cross_valence", "test": f"{r['model']}/{r['direction']}/partial", "p": r["p_partial"]})

# decodability vs behavior: 24 tests
for _, r in sq5.iterrows():
    rows.append({"family": "decod_vs_behavior", "test": f"{r['region']}/{r['decodability']}/{r['behavior']}", "p": r["p"]})

# gradient boosting baseline: 3 metrics x 2 models = 6 tests
for _, r in gbs.iterrows():
    for metric, col in [("raw", "p_raw"), ("partial", "p_partial"), ("interaction", "p_int")]:
        rows.append({"family": "gb_baseline", "test": f"{r['model']}/{metric}", "p": r[col]})

# H4 region combinations: 3 metrics x 7 combos = 21 tests
for _, r in res_h4.iterrows():
    for metric, col in [("raw", "p_raw"), ("partial", "p_partial"), ("interaction", "p_int")]:
        rows.append({"family": "h4_combination", "test": f"{r['regions']}/{metric}", "p": r[col]})

master = pd.DataFrame(rows).dropna(subset=["p"]).reset_index(drop=True)
master["p_fdr_global"] = multipletests(master["p"], method="fdr_bh")[1]
master["sig_global"] = master["p_fdr_global"] < 0.05
master = master.sort_values("p").reset_index(drop=True)

print(f"=== ONE combined FDR grid — total tests: {len(master)} (families: {master['family'].nunique()}) ===")
print(master.round(4).to_string(index=False))

sig = master[master["sig_global"]]
print(f"\n{'='*70}\nSurvived global FDR: {len(sig)}/{len(master)}\n{'='*70}")
print(sig.round(4).to_string(index=False) if len(sig) else "none, no isolated hit survives the combined correction")

print("\n--- per-family FDR vs global FDR ---")
print("Tests significant under per-family FDR that drop out under global FDR:")
old_sig_tests = {"F/M1/interaction_beta", "PO/M1/interaction_beta", "PO/M3/interaction_beta",
                  "PO/M2/partial_corr"}  # tests that were significant under per-family FDR
dropped = master[master["test"].isin(old_sig_tests) & ~master["sig_global"]]
print(dropped.round(4).to_string(index=False) if len(dropped) else "none dropped.")

# channel count check
print("\n" + "="*70 + "\n=== CHANNEL COUNT CHECK: expected 32 channels, how many are in the tensors? ===\n" + "="*70)
_s0 = next(iter(all_subjects_data.values()))
n_each = {reg: _s0["region_tensors"][reg].shape[1] for reg in REGIONS}
n_total = sum(n_each.values())
print(f"channels in region_tensors: {n_each} -> total = {n_total} (expected: 32)")
print(f"missing/excluded: {32 - n_total} channels. Check which ones (reference/EOG/bad-channel rejection?)"
      f" in the preprocessing script and note why they were dropped.")

=== ONE combined FDR grid — total tests: 90 (families: 5) ===
           family                      test      p  p_fdr_global  sig_global
      gb_baseline                   hgb/raw 0.0000        0.0000        True
   h4_combination                F+C+PO/raw 0.0000        0.0000        True
   h4_combination                  F+PO/raw 0.0000        0.0001        True
   h4_combination                   F+C/raw 0.0000        0.0001        True
   h4_combination                  C+PO/raw 0.0000        0.0001        True
   h4_combination                     F/raw 0.0000        0.0001        True
          H3_grid             F/M1/raw_corr 0.0000        0.0001        True
          H3_grid             C/M1/raw_corr 0.0000        0.0004        True
   h4_combination                     C/raw 0.0000        0.0004        True
          H3_grid             F/M2/raw_corr 0.0001        0.0005        True
      gb_baseline                   ada/raw 0.0001        0.0011        True
          H3_g

In [26]:
# split each region/model into within-win and within-loss correlations to see whether the effect is
# a consistent offset (same sign) or an interaction (opposite signs)

diag_rows = []
for reg in REGIONS:
    for tag in ("m1", "m2", "m3"):
        df = get_predictions(reg, tag, 0)
        tb = reward_confound(df, f"{reg}/{tag}", show_rows=False)
        win = tb["within_win_corr"].dropna(); loss = tb["within_loss_corr"].dropna()
        common = tb.dropna(subset=["within_win_corr", "within_loss_corr"])
        diff = common["within_win_corr"] - common["within_loss_corr"]
        t_w, p_w = ttest_1samp(win, 0); t_l, p_l = ttest_1samp(loss, 0)
        t_d, p_d = ttest_1samp(diff, 0)
        same_sign = np.sign(win.mean()) == np.sign(loss.mean())
        diag_rows.append({
            "region": AB[reg], "model": tag.upper(),
            "win_mean": win.mean(), "p_win": p_w,
            "loss_mean": loss.mean(), "p_loss": p_l,
            "win_loss_same_sign": same_sign,
            "diff_mean(win-loss)": diff.mean(), "p_diff": p_d, "n": len(diff)
        })

diag = pd.DataFrame(diag_rows)
for c in ["p_win", "p_loss", "p_diff"]:
    diag[c + "_fdr"] = multipletests(diag[c], method="fdr_bh")[1]
diag["pattern"] = np.where(
    diag["win_loss_same_sign"] & (diag["p_win_fdr"] < 0.05) & (diag["p_loss_fdr"] < 0.05),
    "consistent-offset (both conditions same direction)",
    np.where(diag["p_diff_fdr"] < 0.05, "sign-flip / interaction-driven", "no clear pattern"))

print("=== within_win vs within_loss decomposition — 3 regions × 3 models (FDR within this 9×3=27-test set) ===")
print(diag[["region", "model", "win_mean", "p_win_fdr", "loss_mean", "p_loss_fdr",
            "win_loss_same_sign", "diff_mean(win-loss)", "p_diff_fdr", "pattern"]].round(4).to_string(index=False))

print("\n=== Summary: mechanism for each region/model ===")
for _, r in diag.iterrows():
    print(f"  {r['region']:3s}/{r['model']:2s}: {r['pattern']}")

print("\n--- PO region ---")
po_diag = diag[diag["region"] == "PO"]
print(po_diag[["model", "win_mean", "loss_mean", "win_loss_same_sign", "pattern"]].round(4).to_string(index=False))

loaded frontal_predictions_with_reward.csv          <- INPUT

========================= FRONTAL/M1 =========================
Mean raw_corr          :  0.0797  (t=5.700, p=9.8581e-06, n=23)
Mean corr_with_reward  :  0.1006  (t=6.492, p=1.5700e-06, n=23)
Mean partial_corr      : -0.0098  (t=-0.790, p=4.3811e-01, n=23)
Mean within_win_corr   : -0.0187  (t=-1.079, p=2.9218e-01, n=23)
Mean within_loss_corr  : -0.0038  (t=-0.250, p=8.0469e-01, n=23)
Fisher z-test (partial > 0): t=-0.789, p=4.3856e-01
loaded pred_frontal_m2_seed0.csv                    <- INPUT

========================= FRONTAL/M2 =========================
Mean raw_corr          :  0.0602  (t=4.953, p=5.9030e-05, n=23)
Mean corr_with_reward  :  0.0731  (t=5.673, p=1.0510e-05, n=23)
Mean partial_corr      :  0.0097  (t=0.661, p=5.1524e-01, n=23)
Mean within_win_corr   :  0.0169  (t=0.945, p=3.5501e-01, n=23)
Mean within_loss_corr  :  0.0009  (t=0.056, p=9.5549e-01, n=23)
Fisher z-test (partial > 0): t=0.655, p=5.1898e-01
load

In [27]:
# classify the mechanism from the global-FDR significance of partial_corr and interaction_beta in the H3 grid
# (the win/loss split above is low powered)

sig_lookup = {}
for _, r in master.iterrows():
    if r["family"] != "H3_grid": continue
    reg, model, metric = r["test"].split("/")
    sig_lookup[(reg, model, metric)] = r["sig_global"]

def classify(reg, model):
    partial_sig = sig_lookup.get((reg, model, "partial_corr"), False)
    inter_sig = sig_lookup.get((reg, model, "interaction_beta"), False)
    if partial_sig and inter_sig: return "both significant"
    if partial_sig: return "consistent-offset (partial_corr only)"
    if inter_sig: return "sign-flip / interaction-driven"
    return "neither significant"

clean = grid[["region", "model", "partial_corr", "p_partial_fdr", "interaction_beta", "p_int_fdr"]].copy()
clean["pattern_global_fdr"] = clean.apply(lambda r: classify(r["region"], r["model"]), axis=1)
print("=== Mechanism classification (global FDR) ===")
print(clean.round(4).to_string(index=False))

print("\n--- PO region focus ---")
print(clean[clean["region"] == "PO"].round(4).to_string(index=False))

=== Mechanism classification (global FDR) ===
region model  partial_corr  p_partial_fdr  interaction_beta  p_int_fdr                    pattern_global_fdr
     F    M1       -0.0098         0.5796           -0.0883     0.5109                   neither significant
     F    M2        0.0097         0.5796            0.0122     0.8160                   neither significant
     F    M3       -0.0096         0.5796           -0.0592     0.6914                   neither significant
     C    M1        0.0185         0.5738           -0.2767     0.1847                   neither significant
     C    M2       -0.0051         0.7490           -0.0808     0.2638                   neither significant
     C    M3        0.0089         0.5796           -0.1935     0.1847                   neither significant
    PO    M1       -0.0083         0.5796           -0.4287     0.0122        sign-flip / interaction-driven
    PO    M2       -0.0473         0.0163           -0.0251     0.6914 consistent-

In [28]:
# compact Transformer decoder, same pipeline as the CNN (checkpoint caching, LOSO), only the architecture differs
# Conv1d stem -> learnable positional embedding -> TransformerEncoder -> mean-pool -> linear head

class CompactEEGTransformer(nn.Module):
    def __init__(self, n_channels, n_times, d_model=32, nhead=4, n_layers=2, ff=64, stem_stride=5):
        super().__init__()
        self.stem = nn.Conv1d(n_channels, d_model, kernel_size=25, stride=stem_stride, padding=12)
        seq_len = (n_times + 2 * 12 - 25) // stem_stride + 1
        self.pos = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=ff,
                                           batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.fc = nn.Linear(d_model, 1)
        self.act = nn.GELU()

    def forward(self, x):                     # x: (B, C, T)
        h = self.act(self.stem(x))             # (B, d_model, seq_len)
        h = h.transpose(1, 2) + self.pos        # (B, seq_len, d_model)
        h = self.encoder(h)
        h = h.mean(dim=1)                       # global average pool over time
        return self.fc(h).squeeze(-1)

def fit_predict_fold_trf(train_X, train_y, test_X, ckpt_name, n_epochs=30, lr=1e-3):
    # same as fit_predict_fold but with the Transformer, checkpoints use a separate "trf_" prefix
    model = CompactEEGTransformer(train_X.shape[1], train_X.shape[2]).to(device)
    ck = find(ckpt_name)
    if ck is not None:
        model.load_state_dict(torch.load(ck, map_location=device)); status = "ckpt"
    else:
        if not ALLOW_TRAIN: raise RuntimeError(f"{ckpt_name} is not stored and ALLOW_TRAIN=False")
        loader = DataLoader(EEGRegionDataset(train_X, train_y), batch_size=32, shuffle=True)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        loss_fn = nn.MSELoss(); model.train()
        for _ in range(n_epochs):
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
        out = MODEL_DIR / ckpt_name
        torch.save(model.state_dict(), out); _track(out, upload=False); status = "trained"
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(test_X, dtype=torch.float32).to(device)).cpu().numpy()
    return pred, status

def loso_predict_trf(region, tag="m1", seed=0):
    torch.manual_seed(seed)
    y = RPE[tag]; rows, status_count, t0 = [], {}, time.time()
    for i, te in enumerate(SUBJECTS, 1):
        tr = [s for s in SUBJECTS if s != te]
        Xtr = np.concatenate([all_subjects_data[s]["region_tensors"][region] for s in tr])
        ytr = np.concatenate([y[s] for s in tr])
        Xte = all_subjects_data[te]["region_tensors"][region]
        mu, sd = Xtr.mean(), Xtr.std() + 1e-8
        pred, status = fit_predict_fold_trf((Xtr - mu) / sd, ytr, (Xte - mu) / sd,
                                            f"trf_loso_{region}_{tag}_{te}_seed{seed}.pt")
        status_count[status] = status_count.get(status, 0) + 1
        if status == "trained": print(f"    fold {i}/{len(SUBJECTS)} {te} trained ({time.time()-t0:.0f}s)")
        rt = combined_trials(te)["rt"].values
        n = min(len(pred), len(rt))
        rows.append(pd.DataFrame({"subject": te, "trial": np.arange(n), "actual_rpe": y[te][:n], "predicted_rpe": pred[:n],
                                  "reward": all_subjects_data[te]["reward"][:n], "rt": rt[:n]}))
    print(f"    [TRF {region}/{tag}/seed{seed}] folds: {status_count}")
    return pd.concat(rows, ignore_index=True)

def get_predictions_trf(region, tag="m1", seed=0):
    return cached(f"pred_trf_{region}_{tag}_seed{seed}.csv", lambda: loso_predict_trf(region, tag, seed))

# PO region: M1, M2, M3, CNN vs Transformer
try:
    trf_rows = []
    for tag in ("m1", "m2", "m3"):
        df_trf = get_predictions_trf("parieto_occipital", tag, 0)
        tb = reward_confound(df_trf, f"TRF PO/{tag.upper()}", show_rows=False)
        it = interaction_test(df_trf)
        trf_rows.append({"model": tag.upper(), "raw_corr": tb["raw_corr"].mean(), "p_raw": ttest_1samp(tb["raw_corr"].dropna(), 0)[1],
                         "partial_corr": tb["partial_corr"].mean(), "p_partial": ttest_1samp(tb["partial_corr"].dropna(), 0)[1],
                         "interaction_beta": it["mean_beta"], "p_int": it["p"]})
    trf_df = pd.DataFrame(trf_rows)
    for c in ["p_raw", "p_partial", "p_int"]: trf_df[c + "_fdr"] = multipletests(trf_df[c], method="fdr_bh")[1]

    cnn_po = grid[grid["region"] == "PO"][["model", "raw_corr", "partial_corr", "interaction_beta"]].reset_index(drop=True)
    print("=== Transformer (PO, seed 0) vs CNN results from the H3 grid ===")
    comp = trf_df[["model", "raw_corr", "p_raw_fdr", "partial_corr", "p_partial_fdr", "interaction_beta", "p_int_fdr"]].merge(
        cnn_po, on="model", suffixes=("_TRF", "_CNN"))
    print(comp.round(4).to_string(index=False))
finally:
    export_new_files()

loaded pred_trf_parieto_occipital_m1_seed0.csv      <- INPUT

========================= TRF PO/M1 =========================
Mean raw_corr          :  0.0627  (t=7.072, p=4.2939e-07, n=23)
Mean corr_with_reward  :  0.0659  (t=6.114, p=3.7343e-06, n=23)
Mean partial_corr      :  0.0090  (t=1.007, p=3.2469e-01, n=23)
Mean within_win_corr   :  0.0052  (t=0.318, p=7.5358e-01, n=23)
Mean within_loss_corr  :  0.0150  (t=1.036, p=3.1141e-01, n=23)
Fisher z-test (partial > 0): t=1.008, p=3.2444e-01
loaded pred_trf_parieto_occipital_m2_seed0.csv      <- INPUT

========================= TRF PO/M2 =========================
Mean raw_corr          :  0.0343  (t=3.618, p=1.5239e-03, n=23)
Mean corr_with_reward  :  0.0451  (t=4.442, p=2.0470e-04, n=23)
Mean partial_corr      : -0.0110  (t=-1.138, p=2.6722e-01, n=23)
Mean within_win_corr   : -0.0133  (t=-0.928, p=3.6333e-01, n=23)
Mean within_loss_corr  : -0.0103  (t=-0.973, p=3.4103e-01, n=23)
Fisher z-test (partial > 0): t=-1.139, p=2.6694e-01
loaded

In [29]:
# Transformer multi-seed robustness (PO, M1/M2/M3, 4 seeds), checks that the null result is not a one-seed accident

try:
    trf_seed_rows = []
    for tag in ("m1", "m2", "m3"):
        for seed in [0, 1, 2, 3]:
            df_trf = get_predictions_trf("parieto_occipital", tag, seed)
            tb = reward_confound(df_trf, f"TRF PO/{tag.upper()}/seed{seed}", show_rows=False)
            it = interaction_test(df_trf)
            trf_seed_rows.append({
                "model": tag.upper(), "seed": seed,
                "raw_corr": tb["raw_corr"].mean(),
                "partial_corr": tb["partial_corr"].mean(), "p_partial": ttest_1samp(tb["partial_corr"].dropna(), 0)[1],
                "interaction_beta": it["mean_beta"], "p_int": it["p"]
            })
    trf_seed_df = pd.DataFrame(trf_seed_rows)
    print("=== Transformer PO, results per seed (M1/M2/M3) ===")
    print(trf_seed_df.round(4).to_string(index=False))

    print("\n=== Model-wise summary (mean ± SD across 4 seeds) ===")
    summ_rows = []
    for m, g in trf_seed_df.groupby("model"):
        summ_rows.append({
            "model": m,
            "partial_corr_mean": g["partial_corr"].mean(), "partial_corr_sd": g["partial_corr"].std(),
            "seeds_partial_sig(p<0.05)": int((g["p_partial"] < 0.05).sum()),
            "interaction_beta_mean": g["interaction_beta"].mean(), "interaction_beta_sd": g["interaction_beta"].std(),
            "seeds_interaction_sig(p<0.05)": int((g["p_int"] < 0.05).sum())
        })
    print(pd.DataFrame(summ_rows).round(4).to_string(index=False))

finally:
    export_new_files()

loaded pred_trf_parieto_occipital_m1_seed0.csv      <- INPUT

========================= TRF PO/M1/SEED0 =========================
Mean raw_corr          :  0.0627  (t=7.072, p=4.2939e-07, n=23)
Mean corr_with_reward  :  0.0659  (t=6.114, p=3.7343e-06, n=23)
Mean partial_corr      :  0.0090  (t=1.007, p=3.2469e-01, n=23)
Mean within_win_corr   :  0.0052  (t=0.318, p=7.5358e-01, n=23)
Mean within_loss_corr  :  0.0150  (t=1.036, p=3.1141e-01, n=23)
Fisher z-test (partial > 0): t=1.008, p=3.2444e-01
loaded pred_trf_parieto_occipital_m1_seed1.csv      <- INPUT

========================= TRF PO/M1/SEED1 =========================
Mean raw_corr          :  0.0521  (t=4.749, p=9.6884e-05, n=23)
Mean corr_with_reward  :  0.0607  (t=4.617, p=1.3372e-04, n=23)
Mean partial_corr      :  0.0096  (t=0.857, p=4.0065e-01, n=23)
Mean within_win_corr   :  0.0134  (t=0.889, p=3.8347e-01, n=23)
Mean within_loss_corr  :  0.0026  (t=0.166, p=8.6990e-01, n=23)
Fisher z-test (partial > 0): t=0.858, p=3.9998e-0

In [30]:
# jackknife: drop one subject at a time and redo the group-level interaction test (cached CNN predictions, no training)
try:
    jack_rows = []
    for tag in ("m1", "m3"):   # only M1 and M3, M2 is already null
        df = get_predictions("parieto_occipital", tag, 0)
        all_subs = df["subject"].unique()
        for held_out in all_subs:
            sub_df = df[df["subject"] != held_out]
            r = interaction_test(sub_df)
            jack_rows.append({"model": tag.upper(), "held_out": held_out,
                               "mean_beta": r["mean_beta"], "p": r["p"], "n": r["n"]})
    jack_df = pd.DataFrame(jack_rows)

    print("=== Jackknife: leave one subject out ===")
    for tag in ("M1", "M3"):
        g = jack_df[jack_df["model"] == tag]
        print(f"\n--- {tag} ---")
        print(f"  jackknife β range: [{g['mean_beta'].min():.4f}, {g['mean_beta'].max():.4f}]")
        print(f"  jackknife β SD: {g['mean_beta'].std():.4f}")
        still_sig = (g["p"] < 0.05).sum()
        print(f"  {still_sig}/{len(g)} still p<0.05 with any one subject dropped")
        worst = g.loc[g["mean_beta"].abs().idxmin()]  # weakest case
        print(f"  most influential subject: {worst['held_out']} "
              f"(β={worst['mean_beta']:.4f} without them, p={worst['p']:.4f})")

finally:
    export_new_files()

loaded pred_parieto_occipital_m1_seed0.csv          <- INPUT
loaded pred_parieto_occipital_m3_seed0.csv          <- INPUT
=== Jackknife: leave one subject out ===

--- M1 ---
  jackknife β range: [-0.4693, -0.3570]
  jackknife β SD: 0.0273
  23/23 still p<0.05 with any one subject dropped
  most influential subject: sub-s20 (β=-0.3570 without them, p=0.0033)

--- M3 ---
  jackknife β range: [-0.5327, -0.3872]
  jackknife β SD: 0.0315
  23/23 still p<0.05 with any one subject dropped
  most influential subject: sub-s20 (β=-0.3872 without them, p=0.0018)
No new files were created, everything was loaded from input.


In [32]:
# CNN vs Transformer: PO interaction across 4 seeds (M1, M3)
COLS = ["arch", "model", "seed", "mean_beta", "p", "n"]

print("=" * 70)
print("STEP 1/3 — CNN robustness (4 seeds)")
print("=" * 70)
cnn_rows = []
for tag in ("m1", "m3"):
    for seed in [0, 1, 2, 3]:
        p = find(f"pred_parieto_occipital_{tag}_seed{seed}.csv")
        if p is None:
            print(f"  pred_parieto_occipital_{tag}_seed{seed}.csv not found, skipping")
            continue
        r = interaction_test(pd.read_csv(p))
        cnn_rows.append({"arch": "CNN", "model": tag.upper(), "seed": seed, **r})
cnn_df = pd.DataFrame(cnn_rows)[COLS]
print(cnn_df.round(4).to_string(index=False))

print("\n" + "=" * 70)
print("STEP 2/3 — Transformer robustness (4 seeds)")
print("=" * 70)
trf_rows = []
for tag in ("m1", "m3"):
    for seed in [0, 1, 2, 3]:
        p = find(f"pred_trf_parieto_occipital_{tag}_seed{seed}.csv")
        if p is None:
            print(f"  pred_trf_parieto_occipital_{tag}_seed{seed}.csv not found, skipping")
            continue
        r = interaction_test(pd.read_csv(p))
        trf_rows.append({"arch": "Transformer", "model": tag.upper(), "seed": seed, **r})
trf_df = pd.DataFrame(trf_rows)[COLS]
print(trf_df.round(4).to_string(index=False))

print("\n" + "=" * 70)
print("STEP 3/3 — Jackknife summary (jack_df from the jackknife cell)")
print("=" * 70)
if "jack_df" in globals():
    for tag in ("M1", "M3"):
        g = jack_df[jack_df["model"] == tag]
        still_sig = (g["p"] < 0.05).sum()
        print(f"  {tag}: {still_sig}/{len(g)} still significant with each subject dropped")
else:
    print("  jack_df not found, run the jackknife cell first")

print("\n" + "=" * 70)
print("SUMMARY: CNN vs Transformer (PO interaction)")
print("=" * 70)
combined = pd.concat([cnn_df, trf_df], ignore_index=True)
summary = combined.groupby(["arch", "model"]).agg(
    mean_beta=("mean_beta", "mean"),
    n_sig=("p", lambda x: (x < 0.05).sum()),
    n_total=("p", "count"),
).reset_index()
print(summary.round(4).to_string(index=False))

STEP 1/3 — CNN robustness (4 seeds)
arch model  seed  mean_beta      p  n
 CNN    M1     0    -0.4287 0.0025 23
 CNN    M1     1    -0.3387 0.0151 23
 CNN    M1     2    -0.4135 0.0054 23
 CNN    M1     3    -0.5690 0.0233 23
 CNN    M3     0    -0.4881 0.0027 23
 CNN    M3     1    -0.5330 0.0014 23
 CNN    M3     2    -0.4444 0.0028 23
 CNN    M3     3    -0.2835 0.0712 23

STEP 2/3 — Transformer robustness (4 seeds)
       arch model  seed  mean_beta      p  n
Transformer    M1     0     0.0093 0.8009 23
Transformer    M1     1    -0.0019 0.9084 23
Transformer    M1     2     0.0122 0.4366 23
Transformer    M1     3     0.0102 0.7231 23
Transformer    M3     0    -0.0231 0.1933 23
Transformer    M3     1    -0.0284 0.1948 23
Transformer    M3     2     0.0009 0.9720 23
Transformer    M3     3     0.0197 0.2260 23

STEP 3/3 — Jackknife summary (jack_df from the jackknife cell)
  M1: 23/23 still significant with each subject dropped
  M3: 23/23 still significant with each subject drop

In [33]:
# z-scored interaction beta + permutation test, CNN vs Transformer (30 epochs)
def subj_betas(df, z=False, shuffle=False, rng=None):
    out = []
    for s, g in df.groupby("subject"):
        p = g["predicted_rpe"].values.astype(float); a = g["actual_rpe"].values; r = g["reward"].values.astype(float)
        if r.std() == 0 or p.std() == 0: continue
        if z: p = (p - p.mean()) / p.std()
        if shuffle: p = rng.permutation(p)
        X = np.column_stack([np.ones_like(p), p, r, p * r])
        out.append(np.linalg.lstsq(X, a, rcond=None)[0][3])
    return np.array(out)

rng = np.random.default_rng(0)
for arch, pref in [("CNN", "pred_parieto_occipital"), ("TRF", "pred_trf_parieto_occipital")]:
    for tag in ("m1", "m3"):
        df = pd.read_csv(find(f"{pref}_{tag}_seed0.csv"))
        sd = df.groupby("subject")["predicted_rpe"].std().mean()
        b_raw = subj_betas(df); b_z = subj_betas(df, z=True)
        obs = b_z.mean()
        perm = [subj_betas(df, z=True, shuffle=True, rng=rng).mean() for _ in range(300)]
        p_perm = (np.sum(np.abs(perm) >= abs(obs)) + 1) / 301
        print(f"{arch}/{tag}: pred SD={sd:.4f} | raw β={b_raw.mean():.4f} (p={ttest_1samp(b_raw,0)[1]:.4f}) | "
              f"z-scored β={obs:.4f} (p={ttest_1samp(b_z,0)[1]:.4f}) | permutation p={p_perm:.4f}")

CNN/m1: pred SD=0.0983 | raw β=-0.4287 (p=0.0025) | z-scored β=-0.0200 (p=0.0033) | permutation p=0.0033
CNN/m3: pred SD=0.0872 | raw β=-0.4881 (p=0.0027) | z-scored β=-0.0218 (p=0.0023) | permutation p=0.0033
TRF/m1: pred SD=0.3531 | raw β=0.0093 (p=0.8009) | z-scored β=-0.0015 (p=0.8586) | permutation p=0.8007
TRF/m3: pred SD=0.3671 | raw β=-0.0231 (p=0.1933) | z-scored β=-0.0061 (p=0.3042) | permutation p=0.3621


In [34]:
import copy

def fit_predict_fold_trf_v2(trX, trY, vaX, vaY, teX, n_epochs=80, lr=1e-3, patience=10):
    model = CompactEEGTransformer(trX.shape[1], trX.shape[2]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    loader = DataLoader(EEGRegionDataset(trX, trY), batch_size=32, shuffle=True)
    vaX_t = torch.tensor(vaX, dtype=torch.float32).to(device)
    vaY_t = torch.tensor(vaY, dtype=torch.float32).to(device)
    loss_fn = nn.MSELoss()
    best, best_state, bad, used = 1e9, None, 0, 0
    for ep in range(n_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(vaX_t), vaY_t).item()
        used = ep + 1
        if v < best - 1e-5:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(teX, dtype=torch.float32).to(device)).cpu().numpy()
    return pred, used

def loso_predict_trf_v2(region, tag="m1", seed=0):
    torch.manual_seed(seed)
    y = RPE[tag]; rows, epochs_used, t0 = [], [], time.time()
    for i, te in enumerate(SUBJECTS):
        tr = [s for s in SUBJECTS if s != te]
        va = tr[i % len(tr)]                       # one validation subject for early stopping
        tr = [s for s in tr if s != va]
        Xtr = np.concatenate([all_subjects_data[s]["region_tensors"][region] for s in tr])
        ytr = np.concatenate([y[s] for s in tr])
        Xva = all_subjects_data[va]["region_tensors"][region]; yva = y[va]
        Xte = all_subjects_data[te]["region_tensors"][region]
        mu, sd = Xtr.mean(), Xtr.std() + 1e-8
        pred, used = fit_predict_fold_trf_v2((Xtr-mu)/sd, ytr, (Xva-mu)/sd, yva, (Xte-mu)/sd)
        epochs_used.append(used)
        print(f"    fold {i+1}/{len(SUBJECTS)} {te} | epochs={used} ({time.time()-t0:.0f}s)")
        rt = combined_trials(te)["rt"].values; n = min(len(pred), len(rt))
        rows.append(pd.DataFrame({"subject": te, "trial": np.arange(n), "actual_rpe": y[te][:n],
                                  "predicted_rpe": pred[:n], "reward": all_subjects_data[te]["reward"][:n], "rt": rt[:n]}))
    print(f"    epochs used: mean={np.mean(epochs_used):.1f}, min={min(epochs_used)}, max={max(epochs_used)}")
    return pd.concat(rows, ignore_index=True)

try:
    df80 = cached("pred_trf80_parieto_occipital_m1_seed0.csv",
                  lambda: loso_predict_trf_v2("parieto_occipital", "m1", 0))
    tb = reward_confound(df80, "TRF80 PO/M1", show_rows=False)
    it = interaction_test(df80)
    print("\npred SD:", round(df80.groupby("subject")["predicted_rpe"].std().mean(), 4))
    print("interaction:", {k: round(v, 4) if isinstance(v, float) else v for k, v in it.items()})
finally:
    export_new_files()

loaded pred_trf80_parieto_occipital_m1_seed0.csv    <- INPUT

========================= TRF80 PO/M1 =========================
Mean raw_corr          :  0.1155  (t=5.584, p=1.2971e-05, n=23)
Mean corr_with_reward  :  0.1294  (t=5.173, p=3.4717e-05, n=23)
Mean partial_corr      :  0.0132  (t=1.076, p=2.9365e-01, n=23)
Mean within_win_corr   : -0.0117  (t=-0.662, p=5.1506e-01, n=23)
Mean within_loss_corr  :  0.0484  (t=3.499, p=2.0297e-03, n=23)
Fisher z-test (partial > 0): t=1.077, p=2.9304e-01

pred SD: 0.0791
interaction: {'mean_beta': np.float64(-0.3755), 'p': np.float64(0.0106), 'n': 23}
No new files were created, everything was loaded from input.


In [35]:
# TRF80 (early stopping): M1 seeds 1-3, M2/M3 seed 0
# needs fit_predict_fold_trf_v2 and loso_predict_trf_v2 from the cell above

# each run takes ~17-20 min (~1.5 h in total)
RUNS = [("m1", 1), ("m1", 2), ("m1", 3), ("m2", 0), ("m3", 0)]

try:
    # cache LOSO predictions for every model/seed
    for tag, seed in RUNS:
        cached(
            f"pred_trf80_parieto_occipital_{tag}_seed{seed}.csv",
            lambda tag=tag, seed=seed: loso_predict_trf_v2("parieto_occipital", tag, seed)
        )

    # summary of all TRF80 runs (including M1 seed 0)
    ALL = [("m1", 0), ("m1", 1), ("m1", 2), ("m1", 3), ("m2", 0), ("m3", 0)]
    rows = []

    for tag, seed in ALL:
        df = cached(f"pred_trf80_parieto_occipital_{tag}_seed{seed}.csv", lambda: None)
        tb = reward_confound(df, f"TRF80 PO/{tag.upper()}/s{seed}", show_rows=False)
        it = interaction_test(df)

        rows.append({
            "model": tag.upper(),
            "seed": seed,
            "pred_SD": df.groupby("subject")["predicted_rpe"].std().mean(),
            "raw_corr": tb["raw_corr"].mean(),
            "partial_corr": tb["partial_corr"].mean(),
            "p_partial": ttest_1samp(tb["partial_corr"].dropna(), 0)[1],
            "interaction_beta": it["mean_beta"],
            "p_int": it["p"]
        })

    summ = pd.DataFrame(rows)
    print("\n=== TRF80 (early stopping) — PO summary ===")
    print(summ.round(4).to_string(index=False))

    # how many M1 seeds are significant
    m1 = summ[summ["model"] == "M1"]
    sig_count = int(((m1["interaction_beta"] < 0) & (m1["p_int"] < 0.05)).sum())
    print(f"\nM1: β < 0 and p < 0.05 -> {sig_count}/{len(m1)} seeds")

finally:
    export_new_files()

loaded pred_trf80_parieto_occipital_m1_seed1.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m1_seed2.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m1_seed3.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m2_seed0.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m3_seed0.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m1_seed0.csv    <- INPUT

========================= TRF80 PO/M1/S0 =========================
Mean raw_corr          :  0.1155  (t=5.584, p=1.2971e-05, n=23)
Mean corr_with_reward  :  0.1294  (t=5.173, p=3.4717e-05, n=23)
Mean partial_corr      :  0.0132  (t=1.076, p=2.9365e-01, n=23)
Mean within_win_corr   : -0.0117  (t=-0.662, p=5.1506e-01, n=23)
Mean within_loss_corr  :  0.0484  (t=3.499, p=2.0297e-03, n=23)
Fisher z-test (partial > 0): t=1.077, p=2.9304e-01
loaded pred_trf80_parieto_occipital_m1_seed1.csv    <- INPUT

========================= TRF80 PO/M1/S1 =========================
Mean raw_corr          :  0.1055  (t=6.731, p=9.1520e-07, n=23)
M

In [37]:
RUNS = [("m2", 1), ("m2", 2), ("m2", 3), ("m3", 1), ("m3", 2), ("m3", 3)]
try:
    for tag, seed in RUNS:
        cached(f"pred_trf80_parieto_occipital_{tag}_seed{seed}.csv",
               lambda tag=tag, seed=seed: loso_predict_trf_v2("parieto_occipital", tag, seed))
finally:
    export_new_files()

loaded pred_trf80_parieto_occipital_m2_seed1.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m2_seed2.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m2_seed3.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m3_seed1.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m3_seed2.csv    <- INPUT
loaded pred_trf80_parieto_occipital_m3_seed3.csv    <- INPUT
No new files were created, everything was loaded from input.


In [38]:
def betas(df, z=True):
    out = []
    for _, g in df.groupby("subject"):
        p = g["predicted_rpe"].values.astype(float)
        a = g["actual_rpe"].values.astype(float)
        r = g["reward"].values.astype(float)

        if r.std() == 0 or p.std() == 0:
            continue
        if z:
            p = (p - p.mean()) / p.std()

        X = np.column_stack([np.ones_like(p), p, r, p * r])
        out.append(np.linalg.lstsq(X, a, rcond=None)[0][3])
    return np.array(out)


SETS = {
    "CNN": "pred_parieto_occipital_{t}_seed{s}.csv",
    "TRF30": "pred_trf_parieto_occipital_{t}_seed{s}.csv",
    "TRF80-ES": "pred_trf80_parieto_occipital_{t}_seed{s}.csv",
}

rows = []
for arch, pat in SETS.items():
    for tag in ("m1", "m2", "m3"):
        for seed in range(4):
            f = find(pat.format(t=tag, s=seed))
            if f is None:
                continue

            b = betas(pd.read_csv(f))
            if len(b) == 0:
                continue

            p_val = ttest_1samp(b, 0)[1] if len(b) > 1 else np.nan
            rows.append({"arch": arch, "model": tag.upper(), "seed": seed,
                         "z_beta": b.mean(), "p": p_val})

R = pd.DataFrame(rows)

if not R.empty:
    print("--- Results per seed ---")
    print(R.round(4).to_string(index=False))

    S = (
        R.assign(neg_sig=(R.z_beta < 0) & (R.p < 0.05))
        .groupby(["arch", "model"])
        .agg(seeds=("seed", "count"), mean_z_beta=("z_beta", "mean"), n_neg_sig=("neg_sig", "sum"))
        .reset_index()
    )

    print("\n--- Summary ---")
    print(S.round(4).to_string(index=False))
else:
    print("no matching CSV files found.")

--- Results per seed ---
    arch model  seed  z_beta      p
     CNN    M1     0 -0.0200 0.0033
     CNN    M1     1 -0.0143 0.0474
     CNN    M1     2 -0.0215 0.0045
     CNN    M1     3 -0.0164 0.0136
     CNN    M2     0 -0.0037 0.5015
     CNN    M2     1 -0.0070 0.2938
     CNN    M2     2 -0.0009 0.9063
     CNN    M2     3  0.0026 0.6517
     CNN    M3     0 -0.0218 0.0023
     CNN    M3     1 -0.0253 0.0027
     CNN    M3     2 -0.0228 0.0059
     CNN    M3     3 -0.0121 0.0558
   TRF30    M1     0 -0.0015 0.8586
   TRF30    M1     1  0.0001 0.9900
   TRF30    M1     2  0.0037 0.4648
   TRF30    M1     3  0.0016 0.8314
   TRF30    M2     0 -0.0018 0.7238
   TRF30    M2     1 -0.0025 0.5968
   TRF30    M2     2 -0.0060 0.2800
   TRF30    M2     3  0.0003 0.9675
   TRF30    M3     0 -0.0061 0.3042
   TRF30    M3     1 -0.0083 0.2110
   TRF30    M3     2  0.0022 0.7872
   TRF30    M3     3  0.0067 0.2082
TRF80-ES    M1     0 -0.0207 0.0029
TRF80-ES    M1     1 -0.0236 0.0135
TRF

In [39]:
# paired test of the per-subject interaction beta, M1 vs M2/M3, both architectures
def per_subject_beta(pred_df):
    # returns a Series: subject -> interaction beta (predicted_rpe x reward)
    out = {}
    for subj, sdf in pred_df.groupby("subject"):
        if len(sdf) < 20 or sdf["reward"].nunique() < 2:
            continue
        sdf = sdf.copy()
        sdf["reward"] = sdf["reward"].astype(float)
        sdf["predicted_rpe"] = sdf["predicted_rpe"].astype(float)
        sdf["predicted_rpe"] = (sdf["predicted_rpe"] - sdf["predicted_rpe"].mean()) / sdf["predicted_rpe"].std()
        sdf["actual_rpe"] = sdf["actual_rpe"].astype(float)

        try:
            m = smf.ols("actual_rpe ~ predicted_rpe * reward", data=sdf).fit()
            if "predicted_rpe:reward" in m.params:
                out[subj] = m.params["predicted_rpe:reward"]
        except Exception:
            continue

    return pd.Series(out, dtype=float)


def paired_test(arch, tagA, tagB, seed, fname_pattern):
    pA = find(fname_pattern.format(tag=tagA, seed=seed))
    pB = find(fname_pattern.format(tag=tagB, seed=seed))
    if pA is None or pB is None:
        return None

    bA = per_subject_beta(pd.read_csv(pA))
    bB = per_subject_beta(pd.read_csv(pB))

    common = bA.index.intersection(bB.index)
    if len(common) < 2:
        return None

    diff = bA[common] - bB[common]
    res = ttest_rel(bA[common], bB[common])

    return {
        "arch": arch,
        "comparison": f"{tagA.upper()}-{tagB.upper()}",
        "seed": seed,
        "mean_diff": diff.mean(),
        "t": res.statistic,
        "p": res.pvalue,
        "n": len(common),
    }


results = []

# CNN
for tagA, tagB in [("m1", "m2"), ("m1", "m3")]:
    for seed in [0, 1, 2, 3]:
        r = paired_test("CNN", tagA, tagB, seed, "pred_parieto_occipital_{tag}_seed{seed}.csv")
        if r:
            results.append(r)

# Transformer (early-stopped, 80 epochs)
for tagA, tagB in [("m1", "m2"), ("m1", "m3")]:
    for seed in [0, 1, 2, 3]:
        r = paired_test("TRF80", tagA, tagB, seed, "pred_trf80_parieto_occipital_{tag}_seed{seed}.csv")
        if r:
            results.append(r)

res_df = pd.DataFrame(results)

if not res_df.empty:
    print("=== Paired test: per-subject β, M1 vs M2/M3, both architectures ===")
    print(res_df.round(4).to_string(index=False))

    print("\n=== Summary per (arch, comparison) — combined across seeds ===")
    for (arch, comp), g in res_df.groupby(["arch", "comparison"]):
        n_sig = (g["p"] < 0.05).sum()
        print(
            f"  {arch:10s} {comp:8s}: mean_diff={g['mean_diff'].mean():+.4f} | "
            f"{n_sig}/{len(g)} seeds significant | median p={g['p'].median():.4f}"
        )
else:
    print("no files found or not enough data to compare.")

=== Paired test: per-subject β, M1 vs M2/M3, both architectures ===
 arch comparison  seed  mean_diff       t      p  n
  CNN      M1-M2     0    -0.0163 -2.6159 0.0158 23
  CNN      M1-M2     1    -0.0073 -0.8092 0.4270 23
  CNN      M1-M2     2    -0.0207 -2.9098 0.0081 23
  CNN      M1-M2     3    -0.0190 -2.5007 0.0203 23
  CNN      M1-M3     0     0.0019  0.4557 0.6530 23
  CNN      M1-M3     1     0.0110  2.1260 0.0450 23
  CNN      M1-M3     2     0.0013  0.3065 0.7621 23
  CNN      M1-M3     3    -0.0042 -0.7143 0.4826 23
TRF80      M1-M2     0    -0.0024 -0.2636 0.7945 23
TRF80      M1-M2     1    -0.0221 -1.9504 0.0640 23
TRF80      M1-M2     2    -0.0189 -2.3090 0.0307 23
TRF80      M1-M2     3    -0.0173 -2.2282 0.0364 23
TRF80      M1-M3     0    -0.0115 -2.1694 0.0411 23
TRF80      M1-M3     1    -0.0046 -0.7129 0.4834 23
TRF80      M1-M3     2    -0.0066 -1.2524 0.2236 23
TRF80      M1-M3     3    -0.0128 -2.3679 0.0271 23

=== Summary per (arch, comparison) — combined a

In [41]:
# out-of-sample model comparison (leave-one-block-out per subject): fit on REW and test on PUN, and the reverse.
# Stricter than the in-sample AIC because the parameters are not evaluated on the data they were fit on.

def held_out_nll(model, fit_trials, test_trials, seed=7):
    x, _ = robust_fit(model, fit_trials, n_starts=10, seed=seed)
    c, r = _arr(test_trials)
    nll, _ = run_model(model, x, c, r)
    return nll

def build_oos_comparison():
    rows = []
    for subj in SUBJECTS:
        key_rew, key_pun = f"{subj}_REW", f"{subj}_PUN"
        if key_rew not in final_pipeline_data or key_pun not in final_pipeline_data:
            continue
        rew, pun = final_pipeline_data[key_rew], final_pipeline_data[key_pun]
        for model in ("m1", "m2", "m3"):
            nll_rew_on_pun = held_out_nll(model, fit_trials=rew, test_trials=pun)   # train REW, test PUN
            nll_pun_on_rew = held_out_nll(model, fit_trials=pun, test_trials=rew)   # train PUN, test REW
            rows.append({"subject": subj, "model": model.upper(),
                         "oos_nll_train_REW_test_PUN": nll_rew_on_pun,
                         "oos_nll_train_PUN_test_REW": nll_pun_on_rew,
                         "oos_nll_total": nll_rew_on_pun + nll_pun_on_rew,
                         "n_test_trials": len(pun) + len(rew)})
        print(f"  ✓ {subj} done")
    return pd.DataFrame(rows)

oos_df = cached("oos_model_comparison.csv", build_oos_comparison)

# subject-wise winner: lowest out-of-sample NLL
piv_oos = oos_df.pivot(index="subject", columns="model", values="oos_nll_total")
piv_oos["winner"] = piv_oos.idxmin(axis=1)
print("\n=== Out-of-sample NLL — subject-wise winner counts ===")
print(piv_oos["winner"].value_counts().to_string())

# paired tests: does M2 also win out-of-sample, or was it only in-sample overfitting?
print("\n=== Paired t-tests on out-of-sample NLL (lower = better) ===")
for a, b in [("M1", "M2"), ("M1", "M3"), ("M2", "M3")]:
    d = piv_oos[a] - piv_oos[b]
    t, p = ttest_1samp(d, 0)
    print(f"  {a} vs {b}: mean Δ={d.mean():+.3f} ({a} minus {b}), t={t:.3f}, p={p:.4f}")


export_new_files()

loaded oos_model_comparison.csv                     <- INPUT

=== Out-of-sample NLL — subject-wise winner counts ===
winner
M2    13
M1     6
M3     4

=== Paired t-tests on out-of-sample NLL (lower = better) ===
  M1 vs M2: mean Δ=+3.942 (M1 minus M2), t=1.059, p=0.3011
  M1 vs M3: mean Δ=+5.609 (M1 minus M3), t=1.080, p=0.2917
  M2 vs M3: mean Δ=+1.668 (M2 minus M3), t=0.220, p=0.8281
No new files were created, everything was loaded from input.


In [43]:
# H1 (model mimicry): in-sample AIC vs out-of-sample NLL, same mimicry criterion on both
print("=" * 78)
print("H1 (model mimicry): in-sample vs out-of-sample")
print("=" * 78)

# in-sample, three_way_df from the AIC comparison
in_sample_rows = []
for a, b in [("aic_m1", "aic_m2"), ("aic_m1", "aic_m3"), ("aic_m2", "aic_m3")]:
    d = three_way_df[a] - three_way_df[b]
    t, p = ttest_1samp(d, 0)
    in_sample_rows.append({"comparison": f"{a[-2:].upper()} vs {b[-2:].upper()}",
                           "mean_diff": d.mean(), "mean_abs_diff": d.abs().mean(),
                           "t": t, "p": p, "mimicry(<4 & p>.05)": d.abs().mean() < 4 and p > 0.05})
in_sample_df = pd.DataFrame(in_sample_rows)
print("\n--- IN-SAMPLE (same-block fit + eval, AIC) ---")
print(in_sample_df.round(4).to_string(index=False))

# out-of-sample, piv_oos from the OOS comparison
oos_rows = []
for a, b in [("M1", "M2"), ("M1", "M3"), ("M2", "M3")]:
    d = piv_oos[a] - piv_oos[b]
    t, p = ttest_1samp(d, 0)
    oos_rows.append({"comparison": f"{a} vs {b}",
                     "mean_diff": d.mean(), "mean_abs_diff": d.abs().mean(),
                     "t": t, "p": p, "mimicry(p>.05)": p > 0.05})
oos_df_verdict = pd.DataFrame(oos_rows)
print("\n--- OUT-OF-SAMPLE (train REW -> test PUN, train PUN -> test REW, NLL) ---")
print(oos_df_verdict.round(4).to_string(index=False))

H1 (model mimicry): in-sample vs out-of-sample

--- IN-SAMPLE (same-block fit + eval, AIC) ---
comparison  mean_diff  mean_abs_diff       t      p  mimicry(<4 & p>.05)
  M1 vs M2     8.6841         9.8590  3.6037 0.0008                False
  M1 vs M3    -2.2991         4.3958 -2.3344 0.0241                False
  M2 vs M3   -10.9832        12.0854 -4.2522 0.0001                False

--- OUT-OF-SAMPLE (train REW -> test PUN, train PUN -> test REW, NLL) ---
comparison  mean_diff  mean_abs_diff      t      p  mimicry(p>.05)
  M1 vs M2     3.9415        12.7648 1.0590 0.3011            True
  M1 vs M3     5.6091         8.7155 1.0803 0.2917            True
  M2 vs M3     1.6676        17.5065 0.2197 0.8281            True


In [44]:
# out-of-sample robustness: outlier check + non-parametric test

from scipy.stats import wilcoxon, iqr

print("=" * 78)
print("OOS NLL diff distribution — per-subject spread (outlier check)")
print("=" * 78)
for a, b in [("M1", "M2"), ("M1", "M3"), ("M2", "M3")]:
    d = (piv_oos[a] - piv_oos[b]).dropna()
    print(f"\n--- {a} vs {b} (n={len(d)}) ---")
    print(f"  mean={d.mean():+.3f}  median={d.median():+.3f}  SD={d.std():.3f}  IQR={iqr(d):.3f}")
    print(f"  min={d.min():+.3f}  max={d.max():+.3f}")
    top3 = d.abs().sort_values(ascending=False).head(3)
    print("  subjects with the largest |diff|:")
    for subj, val in top3.items():
        print(f"    {subj}: diff={d[subj]:+.3f}")
    # non-parametric (outlier-robust) test
    try:
        w, p_w = wilcoxon(d)
        print(f"  Wilcoxon signed-rank (robust to outliers): p={p_w:.4f}")
    except Exception as e:
        print(f"  Wilcoxon failed: {e}")
    # t-test again without the 2 most extreme subjects
    trimmed = d[~d.index.isin(top3.index[:2])]
    t2, p2 = ttest_1samp(trimmed, 0)
    print(f"  without the top-2 outliers: mean={trimmed.mean():+.3f}, t={t2:.3f}, p={p2:.4f} (n={len(trimmed)})")

export_new_files()

OOS NLL diff distribution — per-subject spread (outlier check)

--- M1 vs M2 (n=23) ---
  mean=+3.942  median=+1.791  SD=17.850  IQR=17.048
  min=-40.338  max=+40.566
  subjects with the largest |diff|:
    sub-s8: diff=+40.566
    sub-s22: diff=-40.338
    sub-s18: diff=+27.945
  Wilcoxon signed-rank (robust to outliers): p=0.3001
  without the top-2 outliers: mean=+4.306, t=1.450, p=0.1626 (n=21)

--- M1 vs M3 (n=23) ---
  mean=+5.609  median=-0.633  SD=24.900  IQR=2.895
  min=-9.403  max=+114.545
  subjects with the largest |diff|:
    sub-s22: diff=+114.545
    sub-s8: diff=+30.084
    sub-s18: diff=-9.403
  Wilcoxon signed-rank (robust to outliers): p=0.3604
  without the top-2 outliers: mean=-0.744, t=-0.843, p=0.4092 (n=21)

--- M2 vs M3 (n=23) ---
  mean=+1.668  median=-1.584  SD=36.404  IQR=14.898
  min=-37.347  max=+154.883
  subjects with the largest |diff|:
    sub-s22: diff=+154.883
    sub-s18: diff=-37.347
    sub-s13: diff=-31.156
  Wilcoxon signed-rank (robust to outli

In [45]:
# diagnostics for the subjects that keep showing up as out-of-sample outliers

flagged = ["sub-s22", "sub-s18", "sub-s8", "sub-s13"]

print("=" * 78)
print("Flagged subjects — basic diagnostics")
print("=" * 78)
diag_rows = []
for subj in flagged:
    key_rew, key_pun = f"{subj}_REW", f"{subj}_PUN"
    n_rew = len(final_pipeline_data[key_rew]) if key_rew in final_pipeline_data else np.nan
    n_pun = len(final_pipeline_data[key_pun]) if key_pun in final_pipeline_data else np.nan

    row = {"subject": subj, "n_REW": n_rew, "n_PUN": n_pun}

    # in-sample fits (nll per block for M1/M2/M3)
    for model_df, mname in [(final_rw_df, "M1"), (m2_fit_df, "M2"), (m3_fit_df, "M3")]:
        sub_fits = model_df[model_df["subject"] == subj]
        for _, r in sub_fits.iterrows():
            row[f"{mname}_{r['block']}_nll"] = r["nll"]
            row[f"{mname}_{r['block']}_above_chance"] = r["above_chance"]

    # OOS diffs for this subject
    for a, b in [("M1", "M2"), ("M1", "M3"), ("M2", "M3")]:
        if subj in piv_oos.index:
            row[f"OOS_{a}-{b}"] = piv_oos.loc[subj, a] - piv_oos.loc[subj, b]

    diag_rows.append(row)

diag_df = pd.DataFrame(diag_rows)
print(diag_df.round(3).to_string(index=False))

print("\n" + "=" * 78)
print("Comparison with the other 19 subjects (n_trials, above_chance rate)")
print("=" * 78)
rest = [s for s in SUBJECTS if s not in flagged]
n_trials_flagged = [len(final_pipeline_data.get(f"{s}_REW", [])) + len(final_pipeline_data.get(f"{s}_PUN", [])) for s in flagged]
n_trials_rest = [len(final_pipeline_data.get(f"{s}_REW", [])) + len(final_pipeline_data.get(f"{s}_PUN", [])) for s in rest]
print(f"Flagged subjects  mean total trials: {np.mean(n_trials_flagged):.1f}")
print(f"Rest (19 subjects) mean total trials: {np.mean(n_trials_rest):.1f}")

ac_flagged = final_rw_df[final_rw_df["subject"].isin(flagged)]["above_chance"].mean()
ac_rest = final_rw_df[final_rw_df["subject"].isin(rest)]["above_chance"].mean()
print(f"Flagged subjects  M1 above-chance rate: {ac_flagged:.2%}")
print(f"Rest (19 subjects) M1 above-chance rate: {ac_rest:.2%}")

export_new_files()

Flagged subjects — basic diagnostics
subject  n_REW  n_PUN  M1_REW_nll  M1_REW_above_chance  M1_PUN_nll  M1_PUN_above_chance  M2_REW_nll  M2_REW_above_chance  M2_PUN_nll  M2_PUN_above_chance  M3_REW_nll  M3_REW_above_chance  M3_PUN_nll  M3_PUN_above_chance  OOS_M1-M2  OOS_M1-M3  OOS_M2-M3
sub-s22    280    280     124.069                 True      40.940                 True     118.048                 True      39.303                 True     110.880                 True      39.060                 True    -40.338    114.545    154.883
sub-s18    280    280     189.755                False     194.081                False     165.816                 True     194.020                False     183.255                 True     194.075                False     27.945     -9.403    -37.347
 sub-s8    280    280     176.369                 True     194.071                False     164.679                 True     194.020                False     172.921                 True     191.959      

## Status summary

| Stage | Step | Status | Key result |
|---|---|---|---|
| A | NeuroPredict (EEG+ECG) | Complete | External-validated, LOSO, p=0.0016 (n=65) |
| B1 | Data + ground truth | Complete | 23 subjects, 100% match vs the authors' .mat ground truth |
| B2 | RL model fitting (M1/M2/M3) | Complete | Param recovery: M1 r>0.7, M2 good (weak β), M3 weak (α0/κ/η r=0.43-0.56) |
| B3 | Behavioral model comparison | Complete, split-dependent | In-sample AIC: M2 wins 27/46 blocks (p=.0008). Out-of-sample: all 3 models indistinguishable (pairwise p>.29) |
| B4 | EEG to RPE decoding (CNN) | Complete | PO interaction significant for M1 (β=-0.4287, p=.0025) and M3 (β=-0.4881, p=.0027), not M2 |
| B5 | Robustness (RT, seeds, global FDR) | Complete | M1 4/4 seeds, M3 3/4, M2 0/4. PO/M1 and PO/M3 survive global FDR, jackknife 23/23 |
| B6 | Transformer check (30 epochs) | Complete, negative | 0/4 seeds significant for M1 and M3 |
| B8 | Transformer with early stopping (80 epochs) | Complete, confirms B6 | M1 2/4 seeds significant (CNN 4/4), M2/M3 mostly null. Permutation test: CNN p=.003, TRF30 p=.80-.86 |

The PO interaction is specific to the CNN and does not replicate with the Transformer. Behavior favors M2 in-sample while the EEG favors M1/M3, and that mismatch is reported as is.